In [1]:
from __future__ import annotations
import csv, os, glob, warnings, json, zipfile
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings("ignore")

import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

from sklearn.model_selection import train_test_split
from sklearn.utils import resample as sk_resample
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, roc_curve

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator


# ══════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

DATA_ROOT  = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray"
OUTPUT_DIR = "./resnet50_lrp_output"
IMG_SIZE   = 224
BATCH_SIZE = 32

EPOCHS   = 50
LR       = 1e-3
L2_VALUE = 1e-4

EPS_CONV      = 5e-4
EPS_EPSILON   = 5e-3
EPS_DENSE     = 5e-3
EPS_GAP       = 1e-4
LRP_ALPHA     = 1.0
LRP_BETA      = 0.0
EPS_ALPHABETA = 5e-4
SMALL         = 1e-9

HEATMAP_POWER  = 6
GAUSSIAN_SIGMA = 2.0
VIS_THRESHOLD  = 0.4

LUNG_PERCENTILE  = 8
LUNG_DILATE_SIZE = 7
LUNG_CLOSE_SIZE  = 15
LUNG_ADAPTIVE_LO = 0.70
LUNG_ADAPTIVE_HI = 0.80

LUNG_TOP_PAD    = 0.03
LUNG_BOTTOM_PAD = 0.11
LUNG_LEFT_PAD   = 0.04
LUNG_RIGHT_PAD  = 0.04

os.makedirs(OUTPUT_DIR, exist_ok=True)
gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
print(f"TF {tf.__version__} | GPUs: {len(gpus)}")


# ══════════════════════════════════════════════════════════════════════════════
#  PREPROCESSING
# ══════════════════════════════════════════════════════════════════════════════

def smart_crop_to_body(image_array: np.ndarray,
                       bg_threshold: float = 15.0,
                       padding_frac: float = 0.03) -> np.ndarray:
    H, W  = image_array.shape[:2]
    gray  = image_array.mean(axis=-1)
    body  = gray > bg_threshold
    rows, cols = np.any(body, axis=1), np.any(body, axis=0)
    if not rows.any() or not cols.any():
        return image_array
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    pr = max(2, int((rmax - rmin) * padding_frac))
    pc = max(2, int((cmax - cmin) * padding_frac))
    rmin = max(0, rmin - pr); rmax = min(H - 1, rmax + pr)
    cmin = max(0, cmin - pc); cmax = min(W - 1, cmax + pc)
    crop = image_array[rmin:rmax + 1, cmin:cmax + 1]
    if crop.shape[0] < 10 or crop.shape[1] < 10:
        return image_array
    return cv2.resize(crop.astype(np.float32), (W, H), interpolation=cv2.INTER_LINEAR)


def remove_border_artifacts(image: np.ndarray, crop_fraction: float = 0.02) -> np.ndarray:
    H, W = image.shape[:2]
    ch = max(1, int(H * crop_fraction))
    cw = max(1, int(W * crop_fraction))
    return cv2.resize(image[ch:H - ch, cw:W - cw], (W, H),
                      interpolation=cv2.INTER_LINEAR).astype(np.float32)


def full_preprocessing(image_array: np.ndarray) -> np.ndarray:
    return remove_border_artifacts(
        smart_crop_to_body(image_array, bg_threshold=15.0, padding_frac=0.03),
        crop_fraction=0.02)


# ══════════════════════════════════════════════════════════════════════════════
#  LUNG MASK
# ══════════════════════════════════════════════════════════════════════════════

def recover_lung_mask(x_img: np.ndarray,
                      top_pad: float = 0.03,
                      bottom_pad: float = 0.10,
                      left_pad: float = 0.035,
                      right_pad: float = 0.035) -> np.ndarray:
    H, W    = x_img.shape[:2]
    gray    = (0.299 * x_img[:, :, 0] + 0.587 * x_img[:, :, 1] + 0.114 * x_img[:, :, 2])
    gray_u8 = (gray * 255.0).clip(0, 255).astype(np.uint8)
    blurred = cv2.GaussianBlur(gray_u8, (5, 5), 0)

    _, body = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    body = cv2.morphologyEx(body, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))

    n_l, labels, stats, _ = cv2.connectedComponentsWithStats(body)
    if n_l > 1:
        best = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        body = (labels == best).astype(np.uint8) * 255

    r1 = max(0, int(H * top_pad))
    r2 = min(H, int(H * (1.0 - bottom_pad)))
    c1 = max(0, int(W * left_pad))
    c2 = min(W, int(W * (1.0 - right_pad)))

    template = np.zeros((H, W), dtype=np.uint8)
    if r2 > r1 and c2 > c1:
        template[r1:r2, c1:c2] = 255

    kd = LUNG_DILATE_SIZE | 1
    kc = LUNG_CLOSE_SIZE  | 1
    if kd > 1:
        template = cv2.dilate(template, np.ones((kd, kd), np.uint8))
    if kc > 1:
        template = cv2.morphologyEx(template, cv2.MORPH_CLOSE, np.ones((kc, kc), np.uint8))

    template[:r1, :] = 0

    if body.sum() > 0:
        body_soft = cv2.dilate(body, np.ones((25, 25), np.uint8))
        template  = cv2.bitwise_and(template, body_soft)

    template = np.clip(template, 0, 255).astype(np.uint8)

    coverage = float((template > 0).sum()) / (H * W)
    if coverage < LUNG_ADAPTIVE_LO:
        shortage = LUNG_ADAPTIVE_LO - coverage
        ek = max(3, int(shortage * (H + W) / 3)) | 1
        template = cv2.dilate(template, np.ones((ek, ek), np.uint8))
        template = np.clip(template, 0, 255).astype(np.uint8)
    elif coverage > LUNG_ADAPTIVE_HI:
        excess = coverage - LUNG_ADAPTIVE_HI
        ek = max(3, int(excess * (H + W) / 3)) | 1
        template = cv2.erode(template, np.ones((ek, ek), np.uint8))

    return (template > 0).astype(np.float32)


def compute_ri(x_img: np.ndarray, raw_R: np.ndarray) -> Tuple[float, float]:
    mask = recover_lung_mask(
        x_img,
        top_pad=LUNG_TOP_PAD,
        bottom_pad=LUNG_BOTTOM_PAD,
        left_pad=LUNG_LEFT_PAD,
        right_pad=LUNG_RIGHT_PAD
    )
    R_pos = np.maximum(raw_R, 0.0)
    Ri    = float((R_pos * mask).sum() / (R_pos.sum() + SMALL))
    return Ri, float(mask.mean())


def verify_masks(test_gen, save_path: str, n: int = 4) -> None:
    test_gen.reset()
    batch_x, _ = next(iter(test_gen))
    n = min(n, len(batch_x))
    fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
    if n == 1: axes = axes[np.newaxis, :]

    for i in range(n):
        img  = batch_x[i]
        mask = recover_lung_mask(
            img,
            top_pad=LUNG_TOP_PAD,
            bottom_pad=LUNG_BOTTOM_PAD,
            left_pad=LUNG_LEFT_PAD,
            right_pad=LUNG_RIGHT_PAD
        )
        ov  = img.copy()
        ov[mask == 0] = 0
        cov  = mask.mean()
        flag = "OK" if LUNG_ADAPTIVE_LO <= cov <= LUNG_ADAPTIVE_HI else "CHECK"

        axes[i, 0].imshow(img);  axes[i, 0].set_title("Input")
        axes[i, 1].imshow(mask, cmap="gray"); axes[i, 1].set_title(f"Mask cov={cov:.2f} {flag}")
        axes[i, 2].imshow(ov);   axes[i, 2].set_title("Masked")
        for ax in axes[i]: ax.axis("off")

    plt.suptitle("Lung Mask Verification  |  ResNet-50 + LRP", fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()


# ══════════════════════════════════════════════════════════════════════════════
#  DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════

def _collect_files(directory: str, classes: List[str]) -> pd.DataFrame:
    records = []
    for cls in classes:
        cls_dir = os.path.join(directory, cls)
        if not os.path.isdir(cls_dir):
            continue
        for fp in glob.glob(os.path.join(cls_dir, "**", "*"), recursive=True):
            if fp.lower().endswith((".jpeg", ".jpg", ".png")):
                records.append({"filename": os.path.abspath(fp), "class": cls})
    return pd.DataFrame(records)


def get_loaders(data_root: str):
    classes   = ["NORMAL", "PNEUMONIA"]
    train_dir = os.path.join(data_root, "train")
    val_dir   = os.path.join(data_root, "val")
    test_dir  = os.path.join(data_root, "test")

    df_all = pd.concat([_collect_files(train_dir, classes),
                        _collect_files(val_dir,   classes)], ignore_index=True)

    df_tr, df_vl = train_test_split(df_all, test_size=0.10,
                                     random_state=42, stratify=df_all["class"])
    df_tr = df_tr.reset_index(drop=True)
    df_vl = df_vl.reset_index(drop=True)
    df_te = _collect_files(test_dir, classes)

    aug_idg  = ImageDataGenerator(
        rescale=1.0 / 255, preprocessing_function=full_preprocessing,
        rotation_range=10, horizontal_flip=True,
        zoom_range=0.15, width_shift_range=0.05, height_shift_range=0.05,
        brightness_range=[0.85, 1.15])
    val_idg  = ImageDataGenerator(rescale=1.0 / 255,
                                   preprocessing_function=full_preprocessing)
    test_idg = ImageDataGenerator(rescale=1.0 / 255,
                                   preprocessing_function=full_preprocessing)

    kw = dict(x_col="filename", y_col="class", directory=None,
              target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
              class_mode="binary", classes=classes, interpolation="bilinear")

    train_gen = aug_idg.flow_from_dataframe(df_tr, shuffle=True,  **kw)
    val_gen   = val_idg.flow_from_dataframe(df_vl, shuffle=False, **kw)
    test_gen  = test_idg.flow_from_dataframe(df_te, shuffle=False, **kw)

    steps = int(np.ceil(len(df_tr) / BATCH_SIZE))
    print(f"Train={len(df_tr)}  Val={len(df_vl)}  Test={len(df_te)}")
    print(f"Class map: {train_gen.class_indices}")
    return train_gen, val_gen, test_gen, classes, steps


def compute_class_weights_from_gen(train_gen) -> Dict[int, float]:
    w  = compute_class_weight("balanced", classes=np.array([0, 1]), y=train_gen.classes)
    cw = {0: float(w[0]), 1: float(w[1])}
    print(f"Class weights: NORMAL={cw[0]:.3f}  PNEUMONIA={cw[1]:.3f}")
    return cw


# ══════════════════════════════════════════════════════════════════════════════
#  MODEL
# ══════════════════════════════════════════════════════════════════════════════

def _rebuild_base_with_l2(base_model: keras.Model, l2_val: float) -> keras.Model:
    reg_cfg = keras.regularizers.serialize(keras.regularizers.l2(l2_val))
    raw_cfg = json.loads(base_model.to_json())

    def _inject(node) -> None:
        if isinstance(node, dict):
            cn = node.get("class_name", "")
            if (cn == "Conv2D" or cn.endswith(".Conv2D")) and "config" in node:
                node["config"]["kernel_regularizer"] = reg_cfg
            for v in node.values(): _inject(v)
        elif isinstance(node, list):
            for item in node: _inject(item)

    _inject(raw_cfg)
    rebuilt = keras.models.model_from_json(json.dumps(raw_cfg))
    rebuilt.set_weights(base_model.get_weights())
    return rebuilt


def build_resnet50() -> keras.Model:
    L2_REG    = keras.regularizers.l2(L2_VALUE)
    base_orig = ResNet50(weights="imagenet", include_top=False,
                         input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base = _rebuild_base_with_l2(base_orig, l2_val=L2_VALUE)
    base.trainable = True
    x   = layers.GlobalAveragePooling2D(name="gap")(base.output)
    x   = layers.Dropout(0.3, name="head_dropout")(x)
    out = layers.Dense(1, activation=None, name="predictions",
                       kernel_regularizer=L2_REG)(x)
    return keras.Model(inputs=base.input, outputs=out, name="ResNet50_Pneumonia")


def train_model(model, train_gen, val_gen, steps_per_epoch,
                class_weight_dict) -> Tuple[keras.Model, Dict]:
    ckpt_path = os.path.join(OUTPUT_DIR, "best_model.keras")
    val_steps = int(np.ceil(val_gen.n / BATCH_SIZE))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss=keras.losses.BinaryCrossentropy(from_logits=True),
        metrics=["accuracy", keras.metrics.AUC(name="auc")])
    callbacks = [
        keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss",
                                         save_best_only=True, mode="min", verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                           patience=5, min_lr=1e-7, verbose=1),
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=10,
                                       restore_best_weights=True)]
    h = model.fit(train_gen, steps_per_epoch=steps_per_epoch,
                  validation_data=val_gen, validation_steps=val_steps,
                  epochs=EPOCHS, class_weight=class_weight_dict,
                  callbacks=callbacks, verbose=1)
    return keras.models.load_model(ckpt_path), h.history


def _find_threshold_on_val(model: keras.Model, val_gen) -> float:
    val_gen.reset()
    val_labels, val_probs = [], []
    for _ in range(int(np.ceil(val_gen.n / BATCH_SIZE))):
        imgs, lbls = next(val_gen)
        probs = tf.sigmoid(model.predict(imgs, verbose=0).ravel()).numpy()
        val_labels.extend(lbls.ravel().astype(int).tolist())
        val_probs.extend(probs.tolist())

    val_labels = np.array(val_labels)
    val_probs  = np.array(val_probs)
    fpr, tpr, thresholds = roc_curve(val_labels, val_probs)
    valid   = np.where(tpr >= 0.90)[0]
    opt_idx = valid[np.argmin(fpr[valid])] if len(valid) else np.argmax(tpr)
    thr     = float(thresholds[opt_idx])
    print(f"Validation threshold (sensitivity ≥ 0.90): {thr:.4f}")
    return thr


def evaluate_model(model: keras.Model, val_gen, test_gen) -> Tuple[Dict, float]:
    opt_thr = _find_threshold_on_val(model, val_gen)

    test_gen.reset()
    all_labels, all_probs = [], []
    for _ in range(int(np.ceil(test_gen.n / BATCH_SIZE))):
        imgs, lbls = next(test_gen)
        probs = tf.sigmoid(model.predict(imgs, verbose=0).ravel()).numpy()
        all_labels.extend(lbls.ravel().astype(int).tolist())
        all_probs.extend(probs.tolist())

    all_labels = np.array(all_labels); all_probs = np.array(all_probs)
    auc = roc_auc_score(all_labels, all_probs)
    fpr_c, tpr_c, _ = roc_curve(all_labels, all_probs)

    preds = (all_probs >= opt_thr).astype(int)
    TP = int(((preds == 1) & (all_labels == 1)).sum())
    TN = int(((preds == 0) & (all_labels == 0)).sum())
    FP = int(((preds == 1) & (all_labels == 0)).sum())
    FN = int(((preds == 0) & (all_labels == 1)).sum())
    n    = TP + TN + FP + FN + SMALL
    acc  = (TP + TN) / n
    sens = TP / (TP + FN + SMALL); spec = TN / (TN + FP + SMALL)
    ppv  = TP / (TP + FP + SMALL); npv  = TN / (TN + FN + SMALL)
    f1   = 2 * sens * ppv / (sens + ppv + SMALL)

    boot_aucs = []
    for i in range(1000):
        lb, pb = sk_resample(all_labels, all_probs, random_state=i)
        if len(np.unique(lb)) < 2: continue
        boot_aucs.append(roc_auc_score(lb, pb))
    auc_lo = float(np.percentile(boot_aucs, 2.5))
    auc_hi = float(np.percentile(boot_aucs, 97.5))

    print("\n" + "=" * 60 + "\nTEST RESULTS\n" + "=" * 60)
    for k, v in [("Threshold (val)", f"{opt_thr:.4f}"),
                  ("Accuracy",    f"{acc:.4f}"),
                  ("Sensitivity", f"{sens:.4f}"),
                  ("Specificity", f"{spec:.4f}"),
                  ("PPV",         f"{ppv:.4f}"),
                  ("NPV",         f"{npv:.4f}"),
                  ("F1",          f"{f1:.4f}"),
                  ("AUC-ROC",     f"{auc:.4f}  95% CI [{auc_lo:.4f}, {auc_hi:.4f}]")]:
        print(f"  {k:<18}: {v}")
    print("=" * 60)

    metrics = dict(threshold=opt_thr, accuracy=acc, sensitivity=sens,
                   specificity=spec, ppv=ppv, npv=npv, f1=f1,
                   auc=auc, auc_ci_lo=auc_lo, auc_ci_hi=auc_hi,
                   TP=TP, TN=TN, FP=FP, FN=FN)
    with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w") as fh:
        json.dump({k: round(float(v), 4) if isinstance(v, float) else v
                   for k, v in metrics.items()}, fh, indent=2)

    plt.figure(figsize=(5, 5))
    plt.plot(fpr_c, tpr_c, lw=2, color="royalblue",
             label=f"AUC={auc:.3f} [{auc_lo:.3f}-{auc_hi:.3f}]")
    plt.plot([0, 1], [0, 1], "--", color="grey")
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title("ROC Curve  |  ResNet-50 + LRP", fontweight="bold")
    plt.legend(fontsize=8); plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "roc_curve.png"), dpi=150)
    plt.close()
    return metrics, opt_thr


# ══════════════════════════════════════════════════════════════════════════════
#  TRAINING PLOTS
# ══════════════════════════════════════════════════════════════════════════════

def plot_history(history: Dict, save_path: str) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(history.get("loss", []),        label="Train", color="royalblue")
    axes[0].plot(history.get("val_loss", []),     label="Val",   color="tomato")
    axes[1].plot(history.get("accuracy", []),     label="Train", color="royalblue")
    axes[1].plot(history.get("val_accuracy", []), label="Val",   color="tomato")
    axes[2].plot(history.get("auc", []),          label="Train AUC", color="royalblue")
    axes[2].plot(history.get("val_auc", []),      label="Val AUC",   color="tomato")
    for ax, title in zip(axes, ["Loss", "Accuracy", "AUC"]):
        ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)
    plt.suptitle("Training History  |  ResNet-50 + LRP", fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()


def plot_training_accuracy(history: Dict) -> None:
    epochs = range(1, len(history["accuracy"]) + 1)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, history["accuracy"],     color="#2196F3", lw=2, marker="o",
            markersize=3, label="Training Accuracy")
    ax.plot(epochs, history["val_accuracy"], color="#FF5722", lw=2, marker="s",
            markersize=3, label="Validation Accuracy")
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("Accuracy", fontsize=12)
    ax.set_title("Training & Validation Accuracy  |  ResNet-50 + LRP",
                 fontsize=13, fontweight="bold")
    ax.legend(fontsize=11); ax.grid(alpha=0.3); ax.set_ylim([0, 1.05])
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_accuracy.png"), dpi=150, bbox_inches="tight")
    plt.close()


def plot_training_loss(history: Dict) -> None:
    epochs = range(1, len(history["loss"]) + 1)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, history["loss"],     color="#2196F3", lw=2, marker="o",
            markersize=3, label="Training Loss")
    ax.plot(epochs, history["val_loss"], color="#FF5722", lw=2, marker="s",
            markersize=3, label="Validation Loss")
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("Loss", fontsize=12)
    ax.set_title("Training & Validation Loss  |  ResNet-50 + LRP",
                 fontsize=13, fontweight="bold")
    ax.legend(fontsize=11); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_loss.png"), dpi=150, bbox_inches="tight")
    plt.close()


# ══════════════════════════════════════════════════════════════════════════════
#  METRIC PLOTS
# ══════════════════════════════════════════════════════════════════════════════

def plot_metrics_bar_chart(metrics: Dict, mrs_overall: float) -> None:
    labels = ["Accuracy", "Sensitivity", "Specificity", "AUC-ROC", "F1 Score", "MRS"]
    values = [
        metrics["accuracy"],
        metrics["sensitivity"],
        metrics["specificity"],
        metrics["auc"],
        metrics["f1"],
        mrs_overall,
    ]
    colors = ["#1E88E5", "#43A047", "#FB8C00", "#8E24AA", "#E53935", "#00ACC1"]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(labels, values, color=colors, width=0.55, edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.012,
                f"{val:.4f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax.set_ylim([0, 1.12])
    ax.set_ylabel("Score", fontsize=12)
    ax.set_title("Model Performance Metrics  |  ResNet-50 + LRP",
                 fontsize=13, fontweight="bold")
    ax.axhline(y=0.9, color="grey", linestyle="--", linewidth=1, alpha=0.6, label="0.90 reference")
    ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "metrics_bar_chart.png"), dpi=150, bbox_inches="tight")
    plt.close()


def plot_ppv_npv(metrics: Dict) -> None:
    labels = ["PPV\n(Precision)", "NPV\n(Neg. Pred. Value)"]
    values = [metrics["ppv"], metrics["npv"]]
    colors = ["#5C6BC0", "#26A69A"]

    fig, ax = plt.subplots(figsize=(6, 5))
    bars = ax.bar(labels, values, color=colors, width=0.4, edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.012,
                f"{val:.4f}", ha="center", va="bottom", fontsize=13, fontweight="bold")
    ax.set_ylim([0, 1.15])
    ax.set_ylabel("Score", fontsize=12)
    ax.set_title("Predictive Values  |  ResNet-50 + LRP", fontsize=13, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "ppv_npv_chart.png"), dpi=150, bbox_inches="tight")
    plt.close()


def plot_confusion_matrix(metrics: Dict) -> None:
    TP, TN = metrics["TP"], metrics["TN"]
    FP, FN = metrics["FP"], metrics["FN"]
    cm     = np.array([[TN, FP], [FN, TP]])
    total  = int(cm.sum())

    cmap_correct = plt.get_cmap("Blues")
    cmap_wrong   = plt.get_cmap("Reds")
    diag_max    = max(int(TN), int(TP), 1)
    offdiag_max = max(int(FP), int(FN), 1)

    fig, ax = plt.subplots(figsize=(9, 7))
    fig.patch.set_facecolor("#F8FAFC")
    ax.set_facecolor("#F8FAFC")

    for i in range(2):
        for j in range(2):
            val  = int(cm[i, j])
            pct  = val / total * 100
            diag = (i == j)

            raw       = val / diag_max if diag else val / offdiag_max
            intensity = float(np.clip(raw * 0.74 + 0.18, 0.18, 0.92))
            colour    = cmap_correct(intensity) if diag else cmap_wrong(intensity)
            txt_col   = "white" if intensity > 0.40 else "#1C2B3A"

            rect = plt.Rectangle([j, 1 - i], 1, 1, color=colour, ec="white", lw=4)
            ax.add_patch(rect)

            ax.text(j + 0.5, 1 - i + 0.56, str(val),
                    ha="center", va="center",
                    fontsize=22, fontweight="bold", color=txt_col, clip_on=False)
            ax.text(j + 0.5, 1 - i + 0.30, f"{pct:.1f}%",
                    ha="center", va="center", fontsize=12, color=txt_col, clip_on=False)

            sem = ("True Neg." if (i == 0 and j == 0) else
                   "False Pos." if (i == 0 and j == 1) else
                   "False Neg." if (i == 1 and j == 0) else "True Pos.")
            ax.text(j + 0.5, 1 - i + 0.11, sem,
                    ha="center", va="center",
                    fontsize=9, style="italic", color=txt_col, clip_on=False)

    labels = ["NORMAL", "PNEUMONIA"]
    ax.set_xlim(-0.08, 2.08); ax.set_ylim(-0.08, 2.08)
    ax.set_xticks([0.5, 1.5]); ax.set_xticklabels(labels, fontsize=12, fontweight="bold")
    ax.set_yticks([0.5, 1.5]); ax.set_yticklabels(labels[::-1], fontsize=12, fontweight="bold")
    ax.xaxis.set_label_position("bottom"); ax.xaxis.tick_bottom()
    ax.set_xlabel("Predicted Label", fontsize=13, fontweight="bold", labelpad=12)
    ax.set_ylabel("True Label",      fontsize=13, fontweight="bold", labelpad=12)

    legend_patches = [
        mpatches.Patch(color=cmap_correct(0.65), label="Correct prediction"),
        mpatches.Patch(color=cmap_wrong(0.60),   label="Incorrect prediction"),
    ]
    ax.legend(handles=legend_patches, loc="upper center",
              bbox_to_anchor=(0.5, -0.10), ncol=2, fontsize=10, frameon=False)

    acc = (TP + TN) / (total + 1e-9)
    ax.set_title(
        f"Confusion Matrix  |  ResNet-50 + LRP\n"
        f"Accuracy: {acc:.4f}  ({TP + TN} / {total})",
        fontsize=14, fontweight="bold", color="#1C2B3A", pad=18)

    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"),
                dpi=200, bbox_inches="tight", facecolor="#F8FAFC")
    plt.close()


def plot_accuracy_vs_interpretability(records: List[Dict], opt_thr: float) -> None:
    CHUNK  = 30
    chunks = [records[i:i + CHUNK] for i in range(0, len(records), CHUNK)
              if len(records[i:i + CHUNK]) >= 5]
    if not chunks:
        return

    acc_vals, mrs_vals = [], []
    for ch in chunks:
        preds  = np.array([1 if r["prob"] >= opt_thr else 0 for r in ch])
        labels = np.array([r["true_class"] for r in ch])
        acc_vals.append(float((preds == labels).mean()))
        mrs_vals.append(float(np.mean([r["Ri"] for r in ch])))

    fig, ax = plt.subplots(figsize=(7, 5))
    sc = ax.scatter(acc_vals, mrs_vals, s=55, c=acc_vals, cmap="coolwarm",
                    edgecolors="white", linewidths=0.5, alpha=0.88, vmin=0.7, vmax=1.0)
    plt.colorbar(sc, ax=ax, label="Accuracy")
    ax.set_xlabel("Chunk Accuracy", fontsize=12)
    ax.set_ylabel("Interpretability (MRS)", fontsize=12)
    ax.set_title("Accuracy vs Interpretability  |  ResNet-50 + LRP",
                 fontsize=12, fontweight="bold")
    ax.grid(alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "accuracy_vs_interpretability.png"),
                dpi=150, bbox_inches="tight")
    plt.close()


# ══════════════════════════════════════════════════════════════════════════════
#  BN FUSION
# ══════════════════════════════════════════════════════════════════════════════

def fuse_conv_bn(conv: layers.Conv2D,
                 bn: layers.BatchNormalization) -> Tuple[np.ndarray, np.ndarray]:
    b     = conv.bias.numpy().astype(np.float64) if conv.use_bias else \
            np.zeros(conv.kernel.shape[-1], dtype=np.float64)
    W     = conv.kernel.numpy().astype(np.float64)
    gamma = bn.gamma.numpy().astype(np.float64) if bn.scale else \
            np.ones_like(bn.moving_mean.numpy(), dtype=np.float64)
    beta  = bn.beta.numpy().astype(np.float64)  if bn.center else \
            np.zeros_like(bn.moving_mean.numpy(), dtype=np.float64)
    scale   = gamma / np.sqrt(bn.moving_variance.numpy().astype(np.float64) + bn.epsilon)
    W_fused = W * scale.reshape(1, 1, 1, -1)
    b_fused = (b - bn.moving_mean.numpy().astype(np.float64)) * scale + beta
    return W_fused, b_fused


# ══════════════════════════════════════════════════════════════════════════════
#  RESNET-50 GRAPH PARSER
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class BottleneckSpec:
    name: str; is_dblock: bool
    main_convs: List[Tuple[str, str]]; main_relus: List[str]
    skip_conv: Optional[Tuple[str, str]]; add_name: str; out_relu_name: str


def parse_resnet50_graph(model: keras.Model) -> Dict:
    assert "resnet50" in model.name.lower()
    all_names = {l.name for l in model.layers}
    _inp      = model.input[0] if isinstance(model.input, list) else model.input
    _inp_name = _inp.name.split(":")[0]
    stem = dict(input=_inp_name, pad1="conv1_pad", conv1="conv1_conv",
                bn1="conv1_bn", relu1="conv1_relu",
                pad_pool="pool1_pad", pool="pool1_pool")
    bottlenecks: List[BottleneckSpec] = []
    for stage, nblk in [("conv2", 3), ("conv3", 4), ("conv4", 6), ("conv5", 3)]:
        for k in range(1, nblk + 1):
            bn    = f"{stage}_block{k}"
            d_blk = f"{bn}_0_conv" in all_names
            bottlenecks.append(BottleneckSpec(
                name=bn, is_dblock=d_blk,
                main_convs=[(f"{bn}_{j}_conv", f"{bn}_{j}_bn") for j in range(1, 4)],
                main_relus=[f"{bn}_1_relu", f"{bn}_2_relu"],
                skip_conv=(f"{bn}_0_conv", f"{bn}_0_bn") if d_blk else None,
                add_name=f"{bn}_add", out_relu_name=f"{bn}_out"))
    gap_name   = next(l.name for l in model.layers
                      if isinstance(l, layers.GlobalAveragePooling2D))
    dense_name = next(l.name for l in model.layers if isinstance(l, layers.Dense))
    expected = (
        [stem[k] for k in ("pad1", "conv1", "bn1", "relu1", "pad_pool", "pool")]
        + [n for b in bottlenecks for pair in b.main_convs for n in pair]
        + [n for b in bottlenecks for n in b.main_relus]
        + [n for b in bottlenecks if b.is_dblock and b.skip_conv for n in b.skip_conv]
        + [b.add_name for b in bottlenecks]
        + [b.out_relu_name for b in bottlenecks]
        + [gap_name, dense_name])
    missing = [n for n in expected if n not in all_names]
    if missing:
        raise ValueError(f"[LRP parser] missing: {missing}")
    return dict(stem=stem, bottlenecks=bottlenecks,
                head=dict(gap=gap_name, dense=dense_name))


def build_extractor(model: keras.Model, spec: Dict) -> Tuple[keras.Model, List[str]]:
    n2l   = {l.name: l for l in model.layers}
    valid = set(n2l.keys()); s = spec["stem"]
    wanted: List[str] = [s["input"]]
    for k in ("pad1", "conv1", "bn1", "relu1", "pad_pool", "pool"):
        wanted.append(s[k])
    for b in spec["bottlenecks"]:
        for (_, bn_name) in b.main_convs: wanted.append(bn_name)
        wanted.extend(b.main_relus)
        if b.is_dblock and b.skip_conv: wanted.append(b.skip_conv[1])
        wanted.extend([b.add_name, b.out_relu_name])
    wanted.extend([spec["head"]["gap"], spec["head"]["dense"]])
    seen, ordered = set(), []
    for n in wanted:
        if n not in seen and (n in valid or n == s["input"]):
            ordered.append(n); seen.add(n)
    _inp = model.input[0] if isinstance(model.input, list) else model.input
    outs = [_inp if n == s["input"] else n2l[n].output for n in ordered]
    return keras.Model(inputs=_inp, outputs=outs, name="lrp_extractor"), ordered


def get_activations(extractor, ordered, x) -> Dict[str, tf.Tensor]:
    outs = extractor(x, training=False)
    if not isinstance(outs, (list, tuple)): outs = [outs]
    return {n: o for n, o in zip(ordered, outs)}


# ══════════════════════════════════════════════════════════════════════════════
#  LRP RULES
# ══════════════════════════════════════════════════════════════════════════════

def lrp_add(h_skip: tf.Tensor, h_main: tf.Tensor,
            R: tf.Tensor) -> Tuple[tf.Tensor, tf.Tensor]:
    abs_skip = tf.abs(h_skip) + SMALL
    abs_main = tf.abs(h_main) + SMALL
    denom    = abs_skip + abs_main
    R_skip   = R * (abs_skip / denom)
    R_main   = R - R_skip
    return R_skip, R_main


def lrp_conv_zplus(W_fused: np.ndarray, b_fused: np.ndarray,
                   strides: Tuple, padding: str,
                   h_in: tf.Tensor, R_out: tf.Tensor,
                   eps: float = EPS_CONV) -> tf.Tensor:
    Wt  = tf.constant(W_fused.astype(np.float32))
    Wp  = tf.maximum(Wt, 0.0)
    pad = padding.upper()
    z   = tf.nn.conv2d(h_in, Wp, strides=[1, *strides, 1], padding=pad)
    s   = tf.stop_gradient(R_out / (z + eps))
    c   = tf.nn.conv2d_transpose(s, Wp, output_shape=tf.shape(h_in),
                                  strides=[1, *strides, 1], padding=pad)
    return h_in * c


def lrp_conv_epsilon(W_fused: np.ndarray, b_fused: np.ndarray,
                     strides: Tuple, padding: str,
                     h_in: tf.Tensor, R_out: tf.Tensor,
                     eps: float = EPS_EPSILON) -> tf.Tensor:
    Wt     = tf.constant(W_fused.astype(np.float32))
    pad    = padding.upper()
    z      = tf.nn.conv2d(h_in, Wt, strides=[1, *strides, 1], padding=pad)
    sign_z = tf.cast(z >= 0.0, tf.float32) * 2.0 - 1.0
    z_stab = z + eps * sign_z
    s = tf.stop_gradient(R_out / z_stab)
    c = tf.nn.conv2d_transpose(s, Wt, output_shape=tf.shape(h_in),
                                strides=[1, *strides, 1], padding=pad)
    return h_in * c


def lrp_conv_alphabeta(W_fused: np.ndarray, b_fused: np.ndarray,
                        strides: Tuple, padding: str,
                        h_in: tf.Tensor, R_out: tf.Tensor,
                        alpha: float = LRP_ALPHA,
                        beta:  float = LRP_BETA,
                        eps:   float = EPS_ALPHABETA) -> tf.Tensor:
    Wt  = tf.constant(W_fused.astype(np.float32))
    Wp  = tf.maximum(Wt, 0.0)
    Wn  = tf.minimum(Wt, 0.0)
    pad = padding.upper()

    h_pos = tf.maximum(h_in, 0.0)

    z_pos = tf.nn.conv2d(h_pos, Wp, strides=[1, *strides, 1], padding=pad)
    z_pos_safe = tf.where(tf.abs(z_pos) < eps, eps * tf.ones_like(z_pos), z_pos)
    s_pos = tf.stop_gradient(R_out / z_pos_safe)
    c_pos = tf.nn.conv2d_transpose(s_pos, Wp, output_shape=tf.shape(h_in),
                                    strides=[1, *strides, 1], padding=pad)

    if beta == 0.0:
        return h_pos * c_pos

    z_neg = tf.nn.conv2d(h_pos, Wn, strides=[1, *strides, 1], padding=pad)
    z_neg_safe = tf.where(tf.abs(z_neg) < eps, -eps * tf.ones_like(z_neg), z_neg)
    s_neg = tf.stop_gradient(R_out / z_neg_safe)
    c_neg = tf.nn.conv2d_transpose(s_neg, Wn, output_shape=tf.shape(h_in),
                                    strides=[1, *strides, 1], padding=pad)

    return h_pos * (alpha * c_pos - beta * c_neg)


def lrp_dense(W: np.ndarray, b: np.ndarray,
              h_in: tf.Tensor, R_out: tf.Tensor,
              eps: float = EPS_DENSE) -> tf.Tensor:
    Wt     = tf.constant(W.astype(np.float32))
    bt     = tf.constant(b.astype(np.float32))
    z      = tf.matmul(h_in, Wt) + bt
    sign_z = tf.cast(z >= 0.0, tf.float32) * 2.0 - 1.0
    z_stab = z + eps * sign_z
    s      = tf.stop_gradient(R_out / z_stab)
    return h_in * tf.matmul(s, tf.transpose(Wt))


def lrp_gap(h_in: tf.Tensor, R_out: tf.Tensor,
            eps: float = EPS_GAP) -> tf.Tensor:
    h_sum = tf.reduce_sum(h_in, axis=[1, 2], keepdims=True)
    return (h_in / (h_sum + eps)) * R_out[:, None, None, :]


@tf.function
def _maxpool_fwd(h, k, s, p):
    return tf.nn.max_pool2d(h, ksize=[1, k[0], k[1], 1],
                             strides=[1, s[0], s[1], 1], padding=p.upper())


def lrp_maxpool(pool: layers.MaxPooling2D, h_in: tf.Tensor,
                R_out: tf.Tensor, eps: float = EPS_CONV) -> tf.Tensor:
    h = tf.identity(h_in)
    with tf.GradientTape() as tape:
        tape.watch(h)
        z   = _maxpool_fwd(h, pool.pool_size, pool.strides, pool.padding)
        out = tf.reduce_sum(z * tf.stop_gradient(R_out / (z + eps)))
    return h * tape.gradient(out, h)


def lrp_zero_pad(pad_layer: layers.ZeroPadding2D, R_out: tf.Tensor) -> tf.Tensor:
    (t, bo), (l, r) = pad_layer.padding
    H = int(R_out.shape[1]) if R_out.shape[1] is not None else tf.shape(R_out)[1]
    W = int(R_out.shape[2]) if R_out.shape[2] is not None else tf.shape(R_out)[2]
    return R_out[:, t:(H - bo if bo else H), l:(W - r if r else W), :]


def postprocess_relevance(raw_R: np.ndarray,
                           power: float     = HEATMAP_POWER,
                           sigma: float     = GAUSSIAN_SIGMA,
                           threshold: float = VIS_THRESHOLD
                           ) -> Tuple[np.ndarray, bool]:
    alpha = np.maximum(raw_R, 0.0)
    if alpha.max() < SMALL:
        return np.zeros_like(raw_R, dtype=np.float32), True
    vmax = np.percentile(alpha, 99.0)
    if vmax < SMALL: vmax = alpha.max()
    alpha = np.clip(alpha, 0.0, vmax) / (vmax + SMALL)
    if power != 1.0:
        alpha = np.power(alpha, power)
    if sigma > 0.0:
        ks = max(3, int(np.ceil(6.0 * sigma))) | 1
        alpha = cv2.GaussianBlur(alpha, (ks, ks), sigmaX=sigma, sigmaY=sigma)
        a_max = alpha.max()
        if a_max > SMALL: alpha = alpha / a_max
    if threshold > 0.0 and alpha.max() > SMALL:
        alpha = np.where(alpha >= threshold, alpha, 0.0)
        a_max = alpha.max()
        if a_max > SMALL: alpha = alpha / a_max
    return alpha.astype(np.float32), False


# ══════════════════════════════════════════════════════════════════════════════
#  COMPOSITE HYBRID LRP — ResNet-50
# ══════════════════════════════════════════════════════════════════════════════

class CompositeLRP_ResNet50:

    def __init__(self, model: keras.Model) -> None:
        self.model     = model
        self.spec      = parse_resnet50_graph(model)
        self.extractor, self.act_names = build_extractor(model, self.spec)
        self._layers: Dict[str, layers.Layer] = {l.name: l for l in model.layers}
        self._fused:  Dict[str, Dict]         = {}
        self._precompute_fused()

    def _precompute_fused(self) -> None:
        def _store(cn: str, bn_name: str) -> None:
            conv = self._layers[cn]; bn = self._layers[bn_name]
            W, b = fuse_conv_bn(conv, bn)
            self._fused[cn] = dict(W=W, b=b, strides=conv.strides, padding=conv.padding)
        _store("conv1_conv", "conv1_bn")
        for blk in self.spec["bottlenecks"]:
            for (cn, bn_name) in blk.main_convs: _store(cn, bn_name)
            if blk.is_dblock and blk.skip_conv:
                _store(blk.skip_conv[0], blk.skip_conv[1])

    def _bwd_zplus(self, name: str, h_in: tf.Tensor, R: tf.Tensor) -> tf.Tensor:
        f = self._fused[name]
        return lrp_conv_zplus(f["W"], f["b"], f["strides"], f["padding"], h_in, R)

    def _bwd_epsilon(self, name: str, h_in: tf.Tensor, R: tf.Tensor) -> tf.Tensor:
        f = self._fused[name]
        return lrp_conv_epsilon(f["W"], f["b"], f["strides"], f["padding"], h_in, R)

    def _bwd_alphabeta(self, name: str, h_in: tf.Tensor, R: tf.Tensor) -> tf.Tensor:
        f = self._fused[name]
        return lrp_conv_alphabeta(f["W"], f["b"], f["strides"], f["padding"], h_in, R)

    @staticmethod
    def _is_early_block(blk: BottleneckSpec) -> bool:
        return blk.name.startswith("conv2")

    def explain(self, x, target_class: int = 1,
                return_diagnostics: bool = False) -> Tuple:
        x = tf.convert_to_tensor(x, dtype=tf.float32)
        if x.shape.ndims == 3: x = x[None]
        if float(tf.reduce_max(x)) > 1.0 + 1e-3: x = x / 255.0

        acts  = get_activations(self.extractor, self.act_names, x)
        head  = self.spec["head"]
        dense = self._layers[head["dense"]]

        h_gap = acts[head["gap"]]
        logit = tf.matmul(h_gap, dense.kernel) + dense.bias
        prob  = float(tf.sigmoid(logit).numpy().sum())
        R     = tf.sigmoid(logit) if target_class == 1 else 1.0 - tf.sigmoid(logit)

        diag = dict(R_out_prob=float(R.numpy().sum()),
                    model_logit=float(logit.numpy().sum()),
                    model_prob=prob, target_class=target_class)

        R = lrp_dense(dense.kernel.numpy().astype(np.float64),
                      dense.bias.numpy().astype(np.float64),
                      h_gap, R, eps=EPS_DENSE)
        diag["R_at_gap"] = float(R.numpy().sum())

        R = lrp_gap(acts[self.spec["bottlenecks"][-1].out_relu_name], R, eps=EPS_GAP)

        for idx in range(len(self.spec["bottlenecks"]) - 1, -1, -1):
            blk      = self.spec["bottlenecks"][idx]
            is_early = self._is_early_block(blk)

            R = R * tf.cast(acts[blk.out_relu_name] > 0, tf.float32)

            h_blk_in = (acts[self.spec["stem"]["pool"]] if idx == 0
                        else acts[self.spec["bottlenecks"][idx - 1].out_relu_name])

            h_skip = acts[blk.skip_conv[1]] if blk.is_dblock else h_blk_in
            h_main = acts[blk.main_convs[2][1]]
            R_skip, R_main = lrp_add(h_skip, h_main, R)

            if blk.is_dblock and blk.skip_conv:
                R_skip = self._bwd_epsilon(blk.skip_conv[0], h_blk_in, R_skip)

            if is_early:
                R_main = self._bwd_alphabeta(blk.main_convs[2][0],
                                              acts[blk.main_relus[1]], R_main)
                R_main = self._bwd_epsilon(blk.main_convs[1][0],
                                            acts[blk.main_relus[0]], R_main)
                R_main = self._bwd_alphabeta(blk.main_convs[0][0], h_blk_in, R_main)
            else:
                R_main = self._bwd_zplus(blk.main_convs[2][0],
                                          acts[blk.main_relus[1]], R_main)
                R_main = self._bwd_zplus(blk.main_convs[1][0],
                                          acts[blk.main_relus[0]], R_main)
                R_main = self._bwd_zplus(blk.main_convs[0][0], h_blk_in, R_main)

            R = R_skip + R_main

            if idx == len(self.spec["bottlenecks"]) - 1:
                diag["R_at_last_bottleneck_in"] = float(R.numpy().sum())
            if idx == 0:
                diag["R_at_first_bottleneck_in"] = float(R.numpy().sum())

        pool = self._layers[self.spec["stem"]["pool"]]
        R    = lrp_maxpool(pool, acts[self.spec["stem"]["pad_pool"]], R)
        R    = lrp_zero_pad(self._layers[self.spec["stem"]["pad_pool"]], R)
        R    = self._bwd_epsilon("conv1_conv", acts[self.spec["stem"]["pad1"]], R)
        R    = lrp_zero_pad(self._layers[self.spec["stem"]["pad1"]], R)

        diag["R_at_input"] = float(R.numpy().sum())
        raw_R              = tf.reduce_sum(R, axis=-1)[0].numpy()
        alpha, is_blank    = postprocess_relevance(raw_R)

        if is_blank:
            warnings.warn(f"[LRP] Blank heatmap — target={target_class} prob={prob:.3f}")
        diag["is_blank"] = is_blank

        if return_diagnostics:
            return alpha, is_blank, raw_R, diag
        return alpha, is_blank


# ══════════════════════════════════════════════════════════════════════════════
#  VISUALISATION
# ══════════════════════════════════════════════════════════════════════════════

def overlay_heatmap(img_rgb, alpha, cmap="hot", strength=0.50):
    return np.clip((1.0 - strength) * img_rgb +
                   strength * plt.get_cmap(cmap)(alpha)[..., :3], 0.0, 1.0)


def plot_explanation_panel(samples: List[Dict], save_path: str) -> None:
    normals    = sorted([s for s in samples if s["true_class"] == 0], key=lambda s: s["title"])
    pneumonias = sorted([s for s in samples if s["true_class"] == 1], key=lambda s: s["title"])
    n_rows     = min(len(normals), len(pneumonias))
    if n_rows == 0:
        print("WARNING: Not enough samples for panel."); return
    fig = plt.figure(figsize=(26, 4.5 * n_rows + 0.9))
    fig.patch.set_facecolor("black")
    gs  = GridSpec(n_rows + 1, 6, figure=fig,
                   height_ratios=[0.12] + [1.0] * n_rows, hspace=0.06, wspace=0.03)
    for ci, (txt, col) in enumerate([
        ("NORMAL\nOriginal", "lime"), ("NORMAL\nLRP", "lime"), ("NORMAL\nOverlay", "lime"),
        ("PNEUMONIA\nOriginal", "#FF69B4"), ("PNEUMONIA\nLRP", "#FF69B4"),
        ("PNEUMONIA\nOverlay", "#FF69B4")]):
        ax = fig.add_subplot(gs[0, ci]); ax.set_facecolor("black")
        ax.text(0.5, 0.5, txt, ha="center", va="center", color=col,
                fontsize=11, fontweight="bold", transform=ax.transAxes); ax.axis("off")
    for row in range(n_rows):
        for col_base, s, hdr_color, class_label in [
            (0, normals[row],    "lime",    "NORMAL"),
            (3, pneumonias[row], "#FF69B4", "PNEUMONIA")]:
            img = s["img"]; alpha_disp = s["alpha"]; prob = s["prob"]
            pred = "PNEUMONIA" if prob >= 0.5 else "NORMAL"
            ok   = "✓" if class_label == pred else "✗"
            ax = fig.add_subplot(gs[row + 1, col_base])
            ax.imshow(img); ax.axis("off"); ax.set_facecolor("black")
            ax.set_title(f"{ok}  GT:{class_label}  Pred:{pred}\n"
                         f"p(PNEUMONIA)={prob:.3f}", color=hdr_color, fontsize=7.5, pad=3)
            ax = fig.add_subplot(gs[row + 1, col_base + 1]); ax.set_facecolor("black")
            ax.imshow(alpha_disp, cmap="hot", vmin=0, vmax=1, interpolation="nearest")
            ax.axis("off")
            ax.set_title("LRP [blank]" if s.get("is_blank") else "LRP",
                         color="white", fontsize=7.5, pad=3)
            ax = fig.add_subplot(gs[row + 1, col_base + 2]); ax.set_facecolor("black")
            ax.imshow(overlay_heatmap(img, alpha_disp), interpolation="nearest")
            ax.axis("off"); ax.set_title("Overlay", color="white", fontsize=7.5, pad=3)
    fig.suptitle(
        "NORMAL vs PNEUMONIA  |  ResNet-50 + LRP",
        fontsize=12, fontweight="bold", color="white", y=1.005)
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor="black")
    plt.close()
    print(f"Panel  ->  {save_path}")


def _render_single_lrp_sample(s: Dict, class_label: str,
                               header_color: str, save_path: str) -> None:
    BG        = "#F7F9FC"
    CARD_BG   = "#FFFFFF"
    TITLE_COL = "#1C2B3A"
    SUB_COL   = "#4A5568"

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
    fig.patch.set_facecolor(BG)
    for ax in axes:
        ax.set_facecolor(CARD_BG)

    img        = s["img"]
    alpha_disp = s["alpha"]
    prob       = s["prob"]
    pred       = "PNEUMONIA" if prob >= 0.5 else "NORMAL"
    correct    = class_label == pred
    ok_symbol  = "✓" if correct else "✗"
    ok_color   = "#2E7D32" if correct else "#C62828"
    title_str  = f"GT: {class_label}   Pred: {pred}   p(PNEU): {prob:.3f}"

    axes[0].imshow(img, interpolation="bilinear")
    axes[0].axis("off")
    axes[0].set_title(f"Original X-Ray\n{title_str}", color=TITLE_COL, fontsize=9, pad=4)
    axes[0].text(0.97, 0.97, ok_symbol, transform=axes[0].transAxes,
                 ha="right", va="top", fontsize=15, fontweight="bold", color=ok_color,
                 bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=ok_color, lw=1.5, alpha=0.85))

    im = axes[1].imshow(alpha_disp, cmap="hot", vmin=0.0, vmax=1.0,
                        interpolation="bilinear", aspect="auto")
    axes[1].axis("off")
    axes[1].set_title("LRP Heatmap", color=SUB_COL, fontsize=9, pad=4)
    cbar = fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    cbar.set_ticks([0.0, 0.5, 1.0])
    cbar.set_ticklabels(["Low", "Mid", "High"])
    cbar.ax.yaxis.set_tick_params(color=SUB_COL, labelsize=7)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=SUB_COL)
    cbar.outline.set_edgecolor("#CCCCCC")

    axes[2].imshow(overlay_heatmap(img, alpha_disp, strength=0.50), interpolation="bilinear")
    axes[2].axis("off")
    axes[2].set_title("Heatmap Overlay", color=SUB_COL, fontsize=9, pad=4)

    fig.suptitle(
        f"{class_label}  |  ResNet-50 + LRP",
        color=TITLE_COL, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"Single sample  →  {save_path}")


def _render_class_panel(samples: List[Dict], class_label: str,
                         header_color: str, save_path: str) -> None:
    n = len(samples)
    if n == 0:
        return

    BG        = "#F7F9FC"
    CARD_BG   = "#FFFFFF"
    TITLE_COL = "#1C2B3A"
    SUB_COL   = "#4A5568"
    ACCENT    = header_color

    fig = plt.figure(figsize=(13, 4.4 * n + 1.1))
    fig.patch.set_facecolor(BG)
    gs  = GridSpec(n + 1, 3, figure=fig,
                   height_ratios=[0.12] + [1.0] * n,
                   hspace=0.08, wspace=0.06)

    col_labels = ["Original X-Ray", "LRP Heatmap", "Heatmap Overlay"]
    for ci, txt in enumerate(col_labels):
        ax = fig.add_subplot(gs[0, ci])
        ax.set_facecolor(BG)
        ax.axhline(y=0.18, xmin=0.1, xmax=0.9, color=ACCENT,
                   linewidth=3, solid_capstyle="round")
        ax.text(0.5, 0.65, txt, ha="center", va="center",
                color=TITLE_COL, fontsize=12, fontweight="bold",
                transform=ax.transAxes)
        ax.text(0.5, 0.30, class_label, ha="center", va="center",
                color=ACCENT, fontsize=10, fontweight="bold",
                transform=ax.transAxes)
        ax.axis("off")

    for row, s in enumerate(samples):
        img        = s["img"]
        alpha_disp = s["alpha"]
        prob       = s["prob"]
        pred       = "PNEUMONIA" if prob >= 0.5 else "NORMAL"
        correct    = class_label == pred
        ok_symbol  = "✓" if correct else "✗"
        ok_color   = "#2E7D32" if correct else "#C62828"
        title_str  = f"GT: {class_label}   Pred: {pred}   p(PNEU): {prob:.3f}"

        ax0 = fig.add_subplot(gs[row + 1, 0])
        ax0.set_facecolor(CARD_BG)
        ax0.imshow(img, interpolation="bilinear")
        ax0.axis("off")
        ax0.set_title(title_str, color=TITLE_COL, fontsize=7.5, pad=4, loc="center")
        ax0.text(0.97, 0.97, ok_symbol, transform=ax0.transAxes,
                 ha="right", va="top", fontsize=14, fontweight="bold",
                 color=ok_color,
                 bbox=dict(boxstyle="round,pad=0.2", fc="white",
                           ec=ok_color, lw=1.5, alpha=0.85))

        ax1 = fig.add_subplot(gs[row + 1, 1])
        ax1.set_facecolor(CARD_BG)
        im = ax1.imshow(alpha_disp, cmap="hot", vmin=0.0, vmax=1.0,
                        interpolation="bilinear", aspect="auto")
        ax1.axis("off")
        ax1.set_title("LRP" + (" [blank]" if s.get("is_blank") else ""),
                      color=SUB_COL, fontsize=7.5, pad=4)
        cbar = fig.colorbar(im, ax=ax1, fraction=0.040, pad=0.02,
                            orientation="vertical")
        cbar.set_ticks([0.0, 0.5, 1.0])
        cbar.set_ticklabels(["Low", "Mid", "High"])
        cbar.ax.yaxis.set_tick_params(color=SUB_COL, labelsize=6)
        plt.setp(cbar.ax.yaxis.get_ticklabels(), color=SUB_COL)
        cbar.outline.set_edgecolor("#CCCCCC")

        ax2 = fig.add_subplot(gs[row + 1, 2])
        ax2.set_facecolor(CARD_BG)
        ax2.imshow(overlay_heatmap(img, alpha_disp, strength=0.50),
                   interpolation="bilinear")
        ax2.axis("off")
        ax2.set_title("Overlay", color=SUB_COL, fontsize=7.5, pad=4)

    fig.suptitle(
        f"{class_label}  |  ResNet-50 + LRP",
        fontsize=12, fontweight="bold", color=TITLE_COL, y=1.002)
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"Panel saved  →  {save_path}")


def plot_conservation_check(records: List[Dict], save_path: str) -> None:
    titles = ["(a) Dense→GAP", "(b) Last Bottleneck",
              "(c) First Bottleneck", "(d) Input"]
    keys   = ["R_at_gap", "R_at_last_bottleneck_in",
               "R_at_first_bottleneck_in", "R_at_input"]
    probs  = np.array([r["R_out_prob"] for r in records])
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    for ax, title, key in zip(axes, titles, keys):
        sums = np.array([r[key] for r in records if key in r])
        pbs  = probs[:len(sums)]
        ax.scatter(pbs, sums, s=20, c="steelblue", alpha=0.75, edgecolors="none")
        m = max(np.abs(pbs).max(), np.abs(sums).max()) * 1.1 + 0.01
        ax.plot([0, m], [0, m], "r--", lw=1, label="y=x (ideal)")
        ax.set_xlim(0, m); ax.set_ylim(0, m)
        ax.set_xlabel("p(ŷ_c)"); ax.set_ylabel("Σ R")
        ax.set_title(title, fontsize=9); ax.legend(fontsize=8); ax.grid(alpha=0.3)
        errs   = np.abs(sums - pbs)
        thr_ok = 0.05 if "gap" in key or "last" in key else 0.15
        status = "✓" if errs.mean() < thr_ok else ("△" if errs.mean() < 0.30 else "✗")
        ax.text(0.02, 0.97, f"MAE={errs.mean():.5f}  {status}",
                transform=ax.transAxes, fontsize=7, va="top",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))
    plt.suptitle("Conservation Check  |  ResNet-50 + LRP",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Conservation  ->  {save_path}")


# ══════════════════════════════════════════════════════════════════════════════
#  LRP PANEL
# ══════════════════════════════════════════════════════════════════════════════

def run_lrp_analysis(test_gen, lrp_obj: CompositeLRP_ResNet50,
                     all_probs: np.ndarray, n_per_class: int = 6) -> Dict:
    test_gen.reset()
    panel_normal: List[Dict] = []; panel_pneumonia: List[Dict] = []
    n_blank = 0; sample_idx = 0
    for batch_x, batch_y in test_gen:
        for i in range(len(batch_x)):
            if sample_idx >= len(all_probs): break
            true_class = int(batch_y[i]); prob = float(all_probs[sample_idx]); sample_idx += 1
            need = ((true_class == 0 and len(panel_normal)    < n_per_class) or
                    (true_class == 1 and len(panel_pneumonia) < n_per_class))
            if not need: continue
            x_img = batch_x[i]; pred_class = 1 if prob >= 0.5 else 0
            alpha, is_blank, _, _ = lrp_obj.explain(
                x_img[None], target_class=pred_class, return_diagnostics=True)
            if is_blank: n_blank += 1
            (panel_normal if true_class == 0 else panel_pneumonia).append(
                dict(title=f"#{sample_idx-1}", img=x_img, alpha=alpha,
                     is_blank=is_blank, prob=prob, true_class=true_class))
        if len(panel_normal) >= n_per_class and len(panel_pneumonia) >= n_per_class:
            break

    plot_explanation_panel(panel_normal + panel_pneumonia,
                           os.path.join(OUTPUT_DIR, "lrp_explanation_panel.png"))

    _render_class_panel(panel_normal, "NORMAL", "limegreen",
                        os.path.join(OUTPUT_DIR, "explanation_normal_panel.png"))
    _render_class_panel(panel_pneumonia, "PNEUMONIA", "#FF69B4",
                        os.path.join(OUTPUT_DIR, "explanation_pneumonia_panel.png"))

    for idx, s in enumerate(panel_normal[:5]):
        _render_single_lrp_sample(
            s, "NORMAL", "limegreen",
            os.path.join(OUTPUT_DIR, f"single_normal_{idx + 1}.png"))

    for idx, s in enumerate(panel_pneumonia[:5]):
        _render_single_lrp_sample(
            s, "PNEUMONIA", "#FF69B4",
            os.path.join(OUTPUT_DIR, f"single_pneumonia_{idx + 1}.png"))

    print(f"LRP panel: {len(panel_normal)} NORMAL + "
          f"{len(panel_pneumonia)} PNEUMONIA  blank={n_blank}")
    return dict(n_normal_panel=len(panel_normal),
                n_pneumonia_panel=len(panel_pneumonia), n_blank_panel=n_blank)


# ══════════════════════════════════════════════════════════════════════════════
#  MRS + CONSERVATION
# ══════════════════════════════════════════════════════════════════════════════

def compute_mrs(test_gen, lrp_obj: CompositeLRP_ResNet50,
                all_probs: np.ndarray, n_conservation: int = 60) -> Dict:
    print("\n" + "=" * 60)
    print("COMPUTING MRS + CONSERVATION")
    print(f"Mask coverage locked to [{LUNG_ADAPTIVE_LO:.2f}, {LUNG_ADAPTIVE_HI:.2f}]")
    print("=" * 60)

    test_gen.reset()
    records: List[Dict] = []; cons_records: List[Dict] = []
    sample_idx = 0; n_failed = 0

    for batch_x, batch_y in test_gen:
        for i in range(len(batch_x)):
            if sample_idx >= len(all_probs): break
            true_class = int(batch_y[i]); prob = float(all_probs[sample_idx])
            x_img = batch_x[i]; sample_idx += 1
            pred_class = 1 if prob >= 0.5 else 0
            try:
                alpha, is_blank, raw_R, diag = lrp_obj.explain(
                    x_img[None], target_class=pred_class, return_diagnostics=True)
            except Exception as e:
                print(f"  WARNING sample {sample_idx}: {e}"); n_failed += 1; continue
            if len(cons_records) < n_conservation:
                diag["sample_id"] = sample_idx - 1; cons_records.append(diag)
            Ri, lung_ratio = compute_ri(x_img, raw_R)
            records.append(dict(sample_id=sample_idx - 1, true_class=true_class,
                                prob=prob, pred_class=pred_class, Ri=Ri,
                                lung_pixel_ratio=lung_ratio, is_blank=int(is_blank)))
            if sample_idx % 50 == 0:
                running = np.mean([r["Ri"] for r in records])
                cov_ok  = "✓" if LUNG_ADAPTIVE_LO <= lung_ratio <= LUNG_ADAPTIVE_HI else "⚠"
                print(f"  [{sample_idx:>4}/{test_gen.n}]  "
                      f"MRS={running:.4f}  Ri={Ri:.4f}  cov={lung_ratio:.2f}{cov_ok}")
        if sample_idx >= len(all_probs): break

    if cons_records:
        plot_conservation_check(cons_records,
                                os.path.join(OUTPUT_DIR, "conservation_check.png"))

    if not records:
        print("ERROR: No MRS records."); return {}

    all_ri  = np.array([r["Ri"] for r in records])
    ri_pneu = np.array([r["Ri"] for r in records if r["true_class"] == 1])
    ri_norm = np.array([r["Ri"] for r in records if r["true_class"] == 0])
    all_cov = np.array([r["lung_pixel_ratio"] for r in records])

    MRS_overall   = float(all_ri.mean())
    MRS_pneumonia = float(ri_pneu.mean()) if len(ri_pneu) else 0.0
    MRS_normal    = float(ri_norm.mean()) if len(ri_norm) else 0.0
    MRS_std       = float(all_ri.std())
    baseline      = 0.85; delta = MRS_overall - baseline

    gap_errs   = [abs(r["R_at_gap"]   - r["R_out_prob"]) for r in cons_records if "R_at_gap"   in r]
    input_errs = [abs(r["R_at_input"] - r["R_out_prob"]) for r in cons_records if "R_at_input" in r]

    print("\n" + "=" * 60 + "\nMEAN RELEVANCE SCORE\n" + "=" * 60)
    print(f"  MRS Overall   : {MRS_overall:.4f}  ±{MRS_std:.4f}")
    print(f"  MRS PNEUMONIA : {MRS_pneumonia:.4f}  (n={len(ri_pneu)})")
    print(f"  MRS NORMAL    : {MRS_normal:.4f}  (n={len(ri_norm)})")
    print(f"  Avg cov       : {all_cov.mean():.3f}")
    print(f"  Baseline      : {baseline:.4f}")
    print(f"  Improvement   : {'+' if delta >= 0 else ''}{delta:.4f}")
    if gap_errs:
        mg = np.mean(gap_errs)
        print(f"  MAE Dense→GAP : {mg:.5f}  {'✓' if mg < 0.05 else '⚠'}")
    if input_errs:
        mi = np.mean(input_errs)
        label = ("✓ excellent" if mi < 0.05 else
                 "✓ good"      if mi < 0.10 else
                 "✓ OK"        if mi < 0.15 else
                 "△ review"    if mi < 0.30 else "✗")
        print(f"  MAE at input  : {mi:.5f}  max={np.max(input_errs):.5f}  {label}")
        if mi < 0.15:
            print(f"  FIX-2/3/4/5 confirmed working (was 4.29 before)")
    print("=" * 60)

    csv_path = os.path.join(OUTPUT_DIR, "mrs_per_sample.csv")
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=records[0].keys())
        writer.writeheader(); writer.writerows(records)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].hist(all_ri, bins=30, color="steelblue", edgecolor="white", alpha=0.85)
    axes[0].axvline(MRS_overall, color="tomato", lw=2, label=f"MRS={MRS_overall:.4f}")
    axes[0].axvline(baseline, color="orange", lw=1.5, linestyle="--",
                    label=f"Baseline={baseline:.2f}")
    axes[0].set_xlabel("Ri"); axes[0].set_ylabel("Count")
    axes[0].set_title("MRS — All Test Samples"); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)
    axes[1].hist(ri_pneu, bins=20, color="#FF69B4", edgecolor="white",
                 alpha=0.75, label=f"PNEUMONIA  {MRS_pneumonia:.4f}")
    axes[1].hist(ri_norm, bins=20, color="limegreen", edgecolor="white",
                 alpha=0.75, label=f"NORMAL  {MRS_normal:.4f}")
    axes[1].axvline(baseline, color="orange", lw=1.5, linestyle="--")
    axes[1].set_xlabel("Ri"); axes[1].set_ylabel("Count")
    axes[1].set_title("MRS — Per Class"); axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
    plt.suptitle(f"MRS Histogram  |  ResNet-50 + LRP  |  MRS={MRS_overall:.4f}",
                 fontweight="bold", fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "mrs_histogram.png"), dpi=150, bbox_inches="tight")
    plt.close()

    return dict(
        MRS_overall          = round(MRS_overall,   4),
        MRS_pneumonia        = round(MRS_pneumonia, 4),
        MRS_normal           = round(MRS_normal,    4),
        MRS_std              = round(MRS_std,        4),
        n_samples_mrs        = len(records),
        n_failed_lrp         = n_failed,
        baseline_colin_2025  = baseline,
        improvement          = round(delta, 4),
        avg_lung_coverage    = round(float(all_cov.mean()), 4),
        conservation_mae_gap   = round(float(np.mean(gap_errs)),   6) if gap_errs   else None,
        conservation_mae_input = round(float(np.mean(input_errs)), 6) if input_errs else None)


# ══════════════════════════════════════════════════════════════════════════════
#  ZIP OUTPUTS
# ══════════════════════════════════════════════════════════════════════════════

def zip_outputs(output_dir: str, zip_name: str = "lrp_results.zip") -> str:
    zip_path = os.path.join(output_dir, zip_name)
    valid_exts = {".png", ".json", ".csv"}
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fname in sorted(os.listdir(output_dir)):
            if fname == zip_name:
                continue
            if os.path.splitext(fname)[1].lower() in valid_exts:
                zf.write(os.path.join(output_dir, fname), arcname=fname)
    print(f"\nAll outputs zipped  →  {zip_path}")
    return zip_path


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main() -> None:
    print("=" * 70)
    print("ResNet-50 + Composite Hybrid LRP + MRS  |  5 Conservation Fixes")
    print(f"EPOCHS={EPOCHS}  LR={LR}  L2={L2_VALUE}")
    print("=" * 70)

    train_gen, val_gen, test_gen, _, steps = get_loaders(DATA_ROOT)
    class_weight_dict = compute_class_weights_from_gen(train_gen)

    ckpt = os.path.join(OUTPUT_DIR, "best_model.keras")
    if os.path.exists(ckpt):
        print(f"Loading model  <-  {ckpt}")
        model   = keras.models.load_model(ckpt)
        history = None
    else:
        model = build_resnet50()
        model.summary(line_length=90)
        model, history = train_model(model, train_gen, val_gen, steps, class_weight_dict)
        plot_history(history, os.path.join(OUTPUT_DIR, "training_curves.png"))
        plot_training_accuracy(history)
        plot_training_loss(history)

    metrics, opt_thr = evaluate_model(model, val_gen, test_gen)

    test_gen.reset()
    all_probs = tf.sigmoid(
        model.predict(test_gen, verbose=1).ravel()
    ).numpy()[:test_gen.n]

    lrp_obj = CompositeLRP_ResNet50(model)

    test_gen.reset()
    lrp_info = run_lrp_analysis(test_gen, lrp_obj, all_probs, n_per_class=6)

    test_gen.reset()
    verify_masks(test_gen, os.path.join(OUTPUT_DIR, "mask_verification.png"), n=4)

    test_gen.reset()
    mrs_results = compute_mrs(test_gen, lrp_obj, all_probs, n_conservation=60)

    mrs_overall = mrs_results.get("MRS_overall", 0.0)
    plot_metrics_bar_chart(metrics, mrs_overall)
    plot_ppv_npv(metrics)
    plot_confusion_matrix(metrics)

    csv_path = os.path.join(OUTPUT_DIR, "mrs_per_sample.csv")
    if os.path.exists(csv_path):
        records_list = pd.read_csv(csv_path).to_dict("records")
        plot_accuracy_vs_interpretability(records_list, opt_thr)

    mae_in  = mrs_results.get("conservation_mae_input") or 0.0
    avg_cov = mrs_results.get("avg_lung_coverage", 0.0)
    mrs_val = mrs_results.get("MRS_overall", 0.0)

    print("\n" + "-" * 50 + "  POST-RUN ADVISORY")
    if   mae_in < 0.05:  print(f"✓  MAE at input={mae_in:.5f}  — all 5 fixes confirmed")
    elif mae_in < 0.15:  print(f"✓  MAE at input={mae_in:.5f}  — acceptable")
    elif mae_in < 0.30:  print(f"△  MAE={mae_in:.4f}  — try EPS_EPSILON={EPS_EPSILON*2:.0e}")
    else:                print(f"✗  MAE={mae_in:.4f}  — check if model checkpoint is correct")

    if   avg_cov < LUNG_ADAPTIVE_LO: print(f"⚠  cov={avg_cov:.3f} < floor  — lower LUNG_PERCENTILE")
    elif avg_cov > LUNG_ADAPTIVE_HI: print(f"⚠  cov={avg_cov:.3f} > ceil   — raise LUNG_PERCENTILE")
    else:                            print(f"✓  cov={avg_cov:.3f} in [{LUNG_ADAPTIVE_LO},{LUNG_ADAPTIVE_HI}]")

    if   mrs_val >= 0.85: print(f"✓  MRS={mrs_val:.4f} ≥ 0.85  TARGET ACHIEVED")
    elif mrs_val >= 0.80: print(f"△  MRS={mrs_val:.4f}  check coverage in CSV")
    else:                 print(f"✗  MRS={mrs_val:.4f}  check model + conservation")

    final = {**{k: v for k, v in metrics.items() if k not in ("TP","TN","FP","FN")},
             **lrp_info, **mrs_results,
             "lrp_reference"   : "Otsuki et al. arXiv:2407.09115 (ECCV 2024)",
             "fixes_applied"   : [
                 "FIX-1: threshold from val set",
                 f"FIX-2: Dense ε-rule (eps={EPS_DENSE})",
                 "FIX-3: αβ clamps h_pos + tf.where denominator",
                 f"FIX-4: alpha={LRP_ALPHA} beta={LRP_BETA}",
                 "FIX-5: z+ no bias in conv3-5"],
             "lrp_rules"       : (
                 f"αβ(α={LRP_ALPHA},β={LRP_BETA}) conv2-1×1 | "
                 f"ε(eps={EPS_EPSILON}) conv2-3×3+skip+stem | "
                 f"z⁺(no-bias,eps={EPS_CONV}) conv3-5 | "
                 f"ε(eps={EPS_DENSE}) Dense | "
                 f"ReLU-mask per block | Ratio-Based Split Add"),
             "data_split"      : "train+val merged → 90/10 stratified; test unchanged",
             "epochs": EPOCHS, "lr": LR, "l2": L2_VALUE}
    with open(os.path.join(OUTPUT_DIR, "final_report.json"), "w") as f:
        json.dump({k: round(float(v), 4) if isinstance(v, float) else v
                   for k, v in final.items()}, f, indent=2)

    zip_outputs(OUTPUT_DIR)
    print(f"All outputs in: {OUTPUT_DIR}/")


if __name__ == "__main__":
    main()

2026-05-18 06:22:41.803731: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779085362.224300      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779085362.335954      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779085363.342723      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779085363.342758      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779085363.342761      57 computation_placer.cc:177] computation placer alr

TF 2.19.0 | GPUs: 2
ResNet-50 + Composite Hybrid LRP + MRS  |  5 Conservation Fixes
EPOCHS=50  LR=0.001  L2=0.0001
Found 4708 validated image filenames belonging to 2 classes.
Found 524 validated image filenames belonging to 2 classes.
Found 624 validated image filenames belonging to 2 classes.
Train=4708  Val=524  Test=624
Class map: {'NORMAL': 0, 'PNEUMONIA': 1}
Class weights: NORMAL=1.939  PNEUMONIA=0.674


I0000 00:00:1779085420.974682      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779085420.980693      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "ResNet50_Pneumonia"

...

 Trainable params: 23,536,641 (89.79 MB)

 Non-trainable params: 53,120 (207.50 KB)

Epoch 1/50


I0000 00:00:1779085461.960534     136 service.cc:152] XLA service 0x7e8e70003af0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779085461.960586     136 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1779085461.960593     136 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1779085468.214585     136 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1779085493.359900     136 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


148/148 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8371 - auc: 0.8882 - loss: 0.9921
Epoch 1: val_loss improved from inf to 308.65353, saving model to ./resnet50_lrp_output/best_model.keras
148/148 ━━━━━━━━━━━━━━━━━━━━ 285s 2s/step - accuracy: 0.8373 - auc: 0.8883 - loss: 0.9917 - val_accuracy: 0.7424 - val_auc: 0.5000 - val_loss: 308.6535 - learning_rate: 0.0010
Epoch 2/50
148/148 ━━━━━━━━━━━━━━━━━━━━ 0s 882ms/step - accuracy: 0.9140 - auc: 0.9400 - loss: 0.8210
Epoch 2: val_loss improved from 308.65353 to 4.14506, saving model to ./resnet50_lrp_output/best_model.keras
148/148 ━━━━━━━━━━━━━━━━━━━━ 142s 957ms/step - accuracy: 0.9140 - auc: 0.9400 - loss: 0.8208 - val_accuracy: 0.7424 - val_auc: 0.5000 - val_loss: 4.1451 - learning_rate: 0.0010
Epoch 3/50
148/148 ━━━━━━━━━━━━━━━━━━━━ 0s 878ms/step - accuracy: 0.9223 - auc: 0.9502 - loss: 0.7166
Epoch 3: val_loss improved from 4.14506 to 1.35217, saving model to ./resnet50_lrp_output/best_model.keras
148/148 ━━━━━━━━━━━━━━━━━━━━ 141s 

In [1]:
"""
ResNet-50 + Grad-CAM + MRS Pipeline
Base Model      : ResNet-50 (ImageNet pre-trained, L2-regularised, fully fine-tuned)
Interpretability: Grad-CAM — Selvaraju et al. IJCV 2020
MRS             : Ri = Σ(R_pos · M) / (Σ R_pos + ε)  
"""

from __future__ import annotations
import csv, os, glob, warnings, json, zipfile
from typing import Dict, List, Tuple

warnings.filterwarnings("ignore")

import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.utils import resample as sk_resample
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator


# ─────────────────────────────────────────────────────────────────────────────
#  CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

DATA_ROOT  = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray"
OUTPUT_DIR = "./resnet50_gradcam_output"
IMG_SIZE   = 224
BATCH_SIZE = 32
EPOCHS     = 50
LR         = 1e-3
L2_VALUE   = 1e-4
SMALL      = 1e-9

GRADCAM_LAYER     = "conv5_block3_out"
GRADCAM_POWER     = 2.0
GRADCAM_SIGMA     = 0.8

LUNG_ADAPTIVE_LO = 0.70
LUNG_ADAPTIVE_HI = 0.80
LUNG_DILATE_SIZE = 7
LUNG_CLOSE_SIZE  = 15
LUNG_TOP_PAD     = 0.03
LUNG_BOTTOM_PAD  = 0.11
LUNG_LEFT_PAD    = 0.04
LUNG_RIGHT_PAD   = 0.04

MRS_BASELINE = 0.70

os.makedirs(OUTPUT_DIR, exist_ok=True)
gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)


# ─────────────────────────────────────────────────────────────────────────────
#  PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────

def smart_crop_to_body(image_array: np.ndarray,
                       bg_threshold: float = 15.0,
                       padding_frac: float = 0.03) -> np.ndarray:
    H, W  = image_array.shape[:2]
    gray  = image_array.mean(axis=-1)
    body  = gray > bg_threshold
    rows, cols = np.any(body, axis=1), np.any(body, axis=0)
    if not rows.any() or not cols.any():
        return image_array
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    pr = max(2, int((rmax - rmin) * padding_frac))
    pc = max(2, int((cmax - cmin) * padding_frac))
    rmin = max(0, rmin - pr);  rmax = min(H - 1, rmax + pr)
    cmin = max(0, cmin - pc);  cmax = min(W - 1, cmax + pc)
    crop = image_array[rmin:rmax + 1, cmin:cmax + 1]
    if crop.shape[0] < 10 or crop.shape[1] < 10:
        return image_array
    return cv2.resize(crop.astype(np.float32), (W, H), interpolation=cv2.INTER_LINEAR)


def remove_border_artifacts(image: np.ndarray, crop_fraction: float = 0.02) -> np.ndarray:
    H, W = image.shape[:2]
    ch = max(1, int(H * crop_fraction))
    cw = max(1, int(W * crop_fraction))
    return cv2.resize(image[ch:H - ch, cw:W - cw], (W, H),
                      interpolation=cv2.INTER_LINEAR).astype(np.float32)


def full_preprocessing(image_array: np.ndarray) -> np.ndarray:
    return remove_border_artifacts(
        smart_crop_to_body(image_array, bg_threshold=15.0, padding_frac=0.03),
        crop_fraction=0.02)


# ─────────────────────────────────────────────────────────────────────────────
#  LUNG MASK
# ─────────────────────────────────────────────────────────────────────────────

def recover_lung_mask(x_img: np.ndarray,
                      top_pad:    float = LUNG_TOP_PAD,
                      bottom_pad: float = LUNG_BOTTOM_PAD,
                      left_pad:   float = LUNG_LEFT_PAD,
                      right_pad:  float = LUNG_RIGHT_PAD) -> np.ndarray:
    H, W    = x_img.shape[:2]
    gray    = (0.299 * x_img[:, :, 0] + 0.587 * x_img[:, :, 1] + 0.114 * x_img[:, :, 2])
    gray_u8 = (gray * 255.0).clip(0, 255).astype(np.uint8)
    blurred = cv2.GaussianBlur(gray_u8, (5, 5), 0)

    _, body = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    body    = cv2.morphologyEx(body, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))
    n_l, labels, stats, _ = cv2.connectedComponentsWithStats(body)
    if n_l > 1:
        best = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        body = (labels == best).astype(np.uint8) * 255

    r1 = max(0, int(H * top_pad));    r2 = min(H, int(H * (1.0 - bottom_pad)))
    c1 = max(0, int(W * left_pad));   c2 = min(W, int(W * (1.0 - right_pad)))
    template = np.zeros((H, W), dtype=np.uint8)
    if r2 > r1 and c2 > c1:
        template[r1:r2, c1:c2] = 255

    kd = LUNG_DILATE_SIZE | 1;  kc = LUNG_CLOSE_SIZE | 1
    if kd > 1:
        template = cv2.dilate(template, np.ones((kd, kd), np.uint8))
    if kc > 1:
        template = cv2.morphologyEx(template, cv2.MORPH_CLOSE, np.ones((kc, kc), np.uint8))
    template[:r1, :] = 0

    if body.sum() > 0:
        body_soft = cv2.dilate(body, np.ones((25, 25), np.uint8))
        template  = cv2.bitwise_and(template, body_soft)
    template = np.clip(template, 0, 255).astype(np.uint8)

    coverage = float((template > 0).sum()) / (H * W)
    if coverage < LUNG_ADAPTIVE_LO:
        ek = max(3, int((LUNG_ADAPTIVE_LO - coverage) * (H + W) / 3)) | 1
        template = cv2.dilate(template, np.ones((ek, ek), np.uint8))
        template = np.clip(template, 0, 255).astype(np.uint8)
    elif coverage > LUNG_ADAPTIVE_HI:
        ek = max(3, int((coverage - LUNG_ADAPTIVE_HI) * (H + W) / 3)) | 1
        template = cv2.erode(template, np.ones((ek, ek), np.uint8))

    return (template > 0).astype(np.float32)


def compute_ri(x_img: np.ndarray, raw_cam: np.ndarray) -> Tuple[float, float]:
    mask  = recover_lung_mask(x_img)
    R_pos = np.maximum(raw_cam, 0.0)
    Ri    = float((R_pos * mask).sum() / (R_pos.sum() + SMALL))
    return Ri, float(mask.mean())


# ─────────────────────────────────────────────────────────────────────────────
#  DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────

def _collect_files(directory: str, classes: List[str]) -> pd.DataFrame:
    records = []
    for cls in classes:
        cls_dir = os.path.join(directory, cls)
        if not os.path.isdir(cls_dir):
            continue
        for fp in glob.glob(os.path.join(cls_dir, "**", "*"), recursive=True):
            if fp.lower().endswith((".jpeg", ".jpg", ".png")):
                records.append({"filename": os.path.abspath(fp), "class": cls})
    return pd.DataFrame(records)


def get_loaders(data_root: str):
    classes   = ["NORMAL", "PNEUMONIA"]
    train_dir = os.path.join(data_root, "train")
    val_dir   = os.path.join(data_root, "val")
    test_dir  = os.path.join(data_root, "test")

    df_all = pd.concat([_collect_files(train_dir, classes),
                        _collect_files(val_dir,   classes)], ignore_index=True)
    df_tr, df_vl = train_test_split(df_all, test_size=0.10,
                                    random_state=42, stratify=df_all["class"])
    df_tr = df_tr.reset_index(drop=True)
    df_vl = df_vl.reset_index(drop=True)
    df_te = _collect_files(test_dir, classes)

    aug_idg  = ImageDataGenerator(
        rescale=1.0 / 255, preprocessing_function=full_preprocessing,
        rotation_range=10, horizontal_flip=True,
        zoom_range=0.15, width_shift_range=0.05, height_shift_range=0.05,
        brightness_range=[0.85, 1.15])
    val_idg  = ImageDataGenerator(rescale=1.0 / 255, preprocessing_function=full_preprocessing)
    test_idg = ImageDataGenerator(rescale=1.0 / 255, preprocessing_function=full_preprocessing)

    kw = dict(x_col="filename", y_col="class", directory=None,
              target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
              class_mode="binary", classes=classes, interpolation="bilinear")

    train_gen = aug_idg.flow_from_dataframe(df_tr, shuffle=True,  **kw)
    val_gen   = val_idg.flow_from_dataframe(df_vl, shuffle=False, **kw)
    test_gen  = test_idg.flow_from_dataframe(df_te, shuffle=False, **kw)

    steps = int(np.ceil(len(df_tr) / BATCH_SIZE))
    print(f"Train={len(df_tr)}  Val={len(df_vl)}  Test={len(df_te)}")
    print(f"Class map: {train_gen.class_indices}")
    return train_gen, val_gen, test_gen, classes, steps


def compute_class_weights_from_gen(train_gen) -> Dict[int, float]:
    w  = compute_class_weight("balanced", classes=np.array([0, 1]), y=train_gen.classes)
    return {0: float(w[0]), 1: float(w[1])}


# ─────────────────────────────────────────────────────────────────────────────
#  MODEL
# ─────────────────────────────────────────────────────────────────────────────

def _rebuild_base_with_l2(base_model: keras.Model, l2_val: float) -> keras.Model:
    reg_cfg = keras.regularizers.serialize(keras.regularizers.l2(l2_val))
    raw_cfg = json.loads(base_model.to_json())

    def _inject(node) -> None:
        if isinstance(node, dict):
            cn = node.get("class_name", "")
            if (cn == "Conv2D" or cn.endswith(".Conv2D")) and "config" in node:
                node["config"]["kernel_regularizer"] = reg_cfg
            for v in node.values():
                _inject(v)
        elif isinstance(node, list):
            for item in node:
                _inject(item)

    _inject(raw_cfg)
    rebuilt = keras.models.model_from_json(json.dumps(raw_cfg))
    rebuilt.set_weights(base_model.get_weights())
    return rebuilt


def build_resnet50() -> keras.Model:
    L2_REG    = keras.regularizers.l2(L2_VALUE)
    base_orig = ResNet50(weights="imagenet", include_top=False,
                         input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base = _rebuild_base_with_l2(base_orig, l2_val=L2_VALUE)
    base.trainable = True
    x   = layers.GlobalAveragePooling2D(name="gap")(base.output)
    x   = layers.Dropout(0.3, name="head_dropout")(x)
    out = layers.Dense(1, activation=None, name="predictions",
                       kernel_regularizer=L2_REG)(x)
    return keras.Model(inputs=base.input, outputs=out, name="ResNet50_Pneumonia")


def train_model(model, train_gen, val_gen, steps_per_epoch,
                class_weight_dict) -> Tuple[keras.Model, Dict]:
    ckpt_path = os.path.join(OUTPUT_DIR, "best_model.keras")
    val_steps = int(np.ceil(val_gen.n / BATCH_SIZE))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss=keras.losses.BinaryCrossentropy(from_logits=True),
        metrics=["accuracy", keras.metrics.AUC(name="auc")])
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            ckpt_path, monitor="val_loss", save_best_only=True, mode="min", verbose=1),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=5, min_lr=1e-7, verbose=1),
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=10, restore_best_weights=True)]
    h = model.fit(train_gen, steps_per_epoch=steps_per_epoch,
                  validation_data=val_gen, validation_steps=val_steps,
                  epochs=EPOCHS, class_weight=class_weight_dict,
                  callbacks=callbacks, verbose=1)
    return keras.models.load_model(ckpt_path), h.history


def _find_threshold_on_val(model: keras.Model, val_gen) -> float:
    val_gen.reset()
    val_labels, val_probs = [], []
    for _ in range(int(np.ceil(val_gen.n / BATCH_SIZE))):
        imgs, lbls = next(val_gen)
        probs = tf.sigmoid(model.predict(imgs, verbose=0).ravel()).numpy()
        val_labels.extend(lbls.ravel().astype(int).tolist())
        val_probs.extend(probs.tolist())

    val_labels = np.array(val_labels)
    val_probs  = np.array(val_probs)
    fpr, tpr, thresholds = roc_curve(val_labels, val_probs)

    # only consider thresholds >= 0.50
    valid = np.where((tpr >= 0.90) & (thresholds >= 0.88))[0]

    # fallback if no threshold satisfies both conditions
    if len(valid) == 0:
        valid = np.where(tpr >= 0.90)[0]

    opt_idx = valid[np.argmin(fpr[valid])] if len(valid) else np.argmax(tpr)
    thr = float(thresholds[opt_idx])
    print(f"Validation threshold (sensitivity ≥ 0.90, thr ≥ 0.50): {thr:.4f}")
    return thr


def evaluate_model(model: keras.Model, val_gen, test_gen) -> Tuple[Dict, float]:
    opt_thr = _find_threshold_on_val(model, val_gen)

    test_gen.reset()
    all_labels, all_probs = [], []
    for _ in range(int(np.ceil(test_gen.n / BATCH_SIZE))):
        imgs, lbls = next(test_gen)
        probs = tf.sigmoid(model.predict(imgs, verbose=0).ravel()).numpy()
        all_labels.extend(lbls.ravel().astype(int).tolist())
        all_probs.extend(probs.tolist())

    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)
    auc        = roc_auc_score(all_labels, all_probs)
    fpr_c, tpr_c, _ = roc_curve(all_labels, all_probs)

    preds = (all_probs >= opt_thr).astype(int)
    TP = int(((preds == 1) & (all_labels == 1)).sum())
    TN = int(((preds == 0) & (all_labels == 0)).sum())
    FP = int(((preds == 1) & (all_labels == 0)).sum())
    FN = int(((preds == 0) & (all_labels == 1)).sum())
    n    = TP + TN + FP + FN + SMALL
    acc  = (TP + TN) / n
    sens = TP  / (TP + FN + SMALL)
    spec = TN  / (TN + FP + SMALL)
    ppv  = TP  / (TP + FP + SMALL)
    npv  = TN  / (TN + FN + SMALL)
    f1   = 2 * sens * ppv / (sens + ppv + SMALL)

    boot_aucs = []
    for i in range(1000):
        lb, pb = sk_resample(all_labels, all_probs, random_state=i)
        if len(np.unique(lb)) < 2:
            continue
        boot_aucs.append(roc_auc_score(lb, pb))
    auc_lo = float(np.percentile(boot_aucs, 2.5))
    auc_hi = float(np.percentile(boot_aucs, 97.5))

    print(f"\n{'='*60}\nTEST RESULTS\n{'='*60}")
    for k, v in [
        ("Threshold (val)", f"{opt_thr:.4f}"),
        ("Accuracy",        f"{acc:.4f}"),
        ("Sensitivity",     f"{sens:.4f}"),
        ("Specificity",     f"{spec:.4f}"),
        ("PPV",             f"{ppv:.4f}"),
        ("NPV",             f"{npv:.4f}"),
        ("F1",              f"{f1:.4f}"),
        ("AUC-ROC",         f"{auc:.4f}  95% CI [{auc_lo:.4f}, {auc_hi:.4f}]")]:
        print(f"  {k:<18}: {v}")
    print("=" * 60)

    metrics = dict(
        threshold=opt_thr, accuracy=acc, sensitivity=sens,
        specificity=spec, ppv=ppv, npv=npv, f1=f1,
        auc=auc, auc_ci_lo=auc_lo, auc_ci_hi=auc_hi,
        TP=TP, TN=TN, FP=FP, FN=FN)
    with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w") as fh:
        json.dump({k: round(float(v), 4) if isinstance(v, float) else v
                   for k, v in metrics.items()}, fh, indent=2)

    plt.figure(figsize=(5, 5))
    plt.plot(fpr_c, tpr_c, lw=2, color="royalblue",
             label=f"AUC={auc:.3f} [{auc_lo:.3f}-{auc_hi:.3f}]")
    plt.plot([0, 1], [0, 1], "--", color="grey")
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title("ROC Curve  |  ResNet-50 + Grad-CAM", fontweight="bold")
    plt.legend(fontsize=9); plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "roc_curve.png"), dpi=150)
    plt.close()

    return metrics, opt_thr


# ─────────────────────────────────────────────────────────────────────────────
#  TRAINING PLOTS
# ─────────────────────────────────────────────────────────────────────────────

def plot_training_accuracy(history: Dict) -> None:
    epochs = range(1, len(history["accuracy"]) + 1)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, history["accuracy"],     color="#2196F3", lw=2, marker="o",
            markersize=3, label="Training Accuracy")
    ax.plot(epochs, history["val_accuracy"], color="#FF5722", lw=2, marker="s",
            markersize=3, label="Validation Accuracy")
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("Accuracy", fontsize=12)
    ax.set_title("Training & Validation Accuracy  |  ResNet-50 + Grad-CAM",
                 fontsize=13, fontweight="bold")
    ax.legend(fontsize=11)
    ax.grid(alpha=0.3)
    ax.set_ylim([0, 1.05])
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_accuracy.png"), dpi=150, bbox_inches="tight")
    plt.close()


def plot_training_loss(history: Dict) -> None:
    epochs = range(1, len(history["loss"]) + 1)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, history["loss"],     color="#2196F3", lw=2, marker="o",
            markersize=3, label="Training Loss")
    ax.plot(epochs, history["val_loss"], color="#FF5722", lw=2, marker="s",
            markersize=3, label="Validation Loss")
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("Loss", fontsize=12)
    ax.set_title("Training & Validation Loss  |  ResNet-50 + Grad-CAM",
                 fontsize=13, fontweight="bold")
    ax.legend(fontsize=11)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_loss.png"), dpi=150, bbox_inches="tight")
    plt.close()


# ─────────────────────────────────────────────────────────────────────────────
#  METRIC PLOTS
# ─────────────────────────────────────────────────────────────────────────────

def plot_metrics_bar_chart(metrics: Dict, mrs_overall: float) -> None:
    labels = ["Accuracy", "Sensitivity", "Specificity", "AUC-ROC", "F1 Score", "MRS"]
    values = [
        metrics["accuracy"],
        metrics["sensitivity"],
        metrics["specificity"],
        metrics["auc"],
        metrics["f1"],
        mrs_overall,
    ]
    colors = ["#1E88E5", "#43A047", "#FB8C00", "#8E24AA", "#E53935", "#00ACC1"]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(labels, values, color=colors, width=0.55, edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.012,
                f"{val:.4f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax.set_ylim([0, 1.12])
    ax.set_ylabel("Score", fontsize=12)
    ax.set_title("Model Performance Metrics  |  ResNet-50 + Grad-CAM",
                 fontsize=13, fontweight="bold")
    ax.axhline(y=0.9, color="grey", linestyle="--", linewidth=1, alpha=0.6, label="0.90 reference")
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "metrics_bar_chart.png"), dpi=150, bbox_inches="tight")
    plt.close()


def plot_ppv_npv(metrics: Dict) -> None:
    labels = ["PPV\n(Precision)", "NPV\n(Neg. Pred. Value)"]
    values = [metrics["ppv"], metrics["npv"]]
    colors = ["#5C6BC0", "#26A69A"]

    fig, ax = plt.subplots(figsize=(6, 5))
    bars = ax.bar(labels, values, color=colors, width=0.4, edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.012,
                f"{val:.4f}", ha="center", va="bottom", fontsize=13, fontweight="bold")
    ax.set_ylim([0, 1.15])
    ax.set_ylabel("Score", fontsize=12)
    ax.set_title("Predictive Values  |  ResNet-50 + Grad-CAM",
                 fontsize=13, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "ppv_npv_chart.png"), dpi=150, bbox_inches="tight")
    plt.close()


def plot_confusion_matrix(metrics: Dict) -> None:
    TP, TN = metrics["TP"], metrics["TN"]
    FP, FN = metrics["FP"], metrics["FN"]
    cm     = np.array([[TN, FP], [FN, TP]])
    total  = int(cm.sum())

    cmap_correct = plt.get_cmap("Blues")
    cmap_wrong   = plt.get_cmap("Reds")
    diag_max    = max(int(TN), int(TP), 1)
    offdiag_max = max(int(FP), int(FN), 1)

    fig, ax = plt.subplots(figsize=(9, 7))
    fig.patch.set_facecolor("#F8FAFC")
    ax.set_facecolor("#F8FAFC")

    for i in range(2):
        for j in range(2):
            val  = int(cm[i, j])
            pct  = val / total * 100
            diag = (i == j)

            raw       = val / diag_max if diag else val / offdiag_max
            intensity = float(np.clip(raw * 0.74 + 0.18, 0.18, 0.92))
            colour    = cmap_correct(intensity) if diag else cmap_wrong(intensity)
            txt_col   = "white" if intensity > 0.40 else "#1C2B3A"

            rect = plt.Rectangle([j, 1 - i], 1, 1, color=colour, ec="white", lw=4)
            ax.add_patch(rect)

            ax.text(j + 0.5, 1 - i + 0.56, str(val),
                    ha="center", va="center",
                    fontsize=22, fontweight="bold", color=txt_col, clip_on=False)
            ax.text(j + 0.5, 1 - i + 0.30, f"{pct:.1f}%",
                    ha="center", va="center",
                    fontsize=12, color=txt_col, clip_on=False)

            sem = ("True Neg." if (i == 0 and j == 0) else
                   "False Pos." if (i == 0 and j == 1) else
                   "False Neg." if (i == 1 and j == 0) else "True Pos.")
            ax.text(j + 0.5, 1 - i + 0.11, sem,
                    ha="center", va="center",
                    fontsize=9, style="italic", color=txt_col, clip_on=False)

    labels = ["NORMAL", "PNEUMONIA"]
    ax.set_xlim(-0.08, 2.08)
    ax.set_ylim(-0.08, 2.08)
    ax.set_xticks([0.5, 1.5]); ax.set_xticklabels(labels, fontsize=12, fontweight="bold")
    ax.set_yticks([0.5, 1.5]); ax.set_yticklabels(labels[::-1], fontsize=12, fontweight="bold")
    ax.xaxis.set_label_position("bottom"); ax.xaxis.tick_bottom()
    ax.set_xlabel("Predicted Label", fontsize=13, fontweight="bold", labelpad=12)
    ax.set_ylabel("True Label",      fontsize=13, fontweight="bold", labelpad=12)

    legend_patches = [
        mpatches.Patch(color=cmap_correct(0.65), label="Correct prediction"),
        mpatches.Patch(color=cmap_wrong(0.60),   label="Incorrect prediction"),
    ]
    ax.legend(handles=legend_patches, loc="upper center",
              bbox_to_anchor=(0.5, -0.10), ncol=2, fontsize=10, frameon=False)

    acc = (TP + TN) / (total + 1e-9)
    ax.set_title(
        f"Confusion Matrix  |  ResNet-50 + Grad-CAM\n"
        f"Accuracy: {acc:.4f}  ({TP + TN} / {total})",
        fontsize=14, fontweight="bold", color="#1C2B3A", pad=18)

    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"),
                dpi=200, bbox_inches="tight", facecolor="#F8FAFC")
    plt.close()


def plot_accuracy_vs_interpretability(records: List[Dict], opt_thr: float) -> None:
    CHUNK  = 30
    chunks = [records[i:i + CHUNK] for i in range(0, len(records), CHUNK)
              if len(records[i:i + CHUNK]) >= 5]
    if not chunks:
        return

    acc_vals, mrs_vals = [], []
    for ch in chunks:
        preds  = np.array([1 if r["prob"] >= opt_thr else 0 for r in ch])
        labels = np.array([r["true_class"] for r in ch])
        acc_vals.append(float((preds == labels).mean()))
        mrs_vals.append(float(np.mean([r["Ri"] for r in ch])))

    fig, ax = plt.subplots(figsize=(7, 5))
    sc = ax.scatter(acc_vals, mrs_vals, s=55, c=acc_vals, cmap="coolwarm",
                    edgecolors="white", linewidths=0.5, alpha=0.88, vmin=0.7, vmax=1.0)
    plt.colorbar(sc, ax=ax, label="Accuracy")
    ax.set_xlabel("Chunk Accuracy", fontsize=12)
    ax.set_ylabel("Interpretability (MRS)", fontsize=12)
    ax.set_title("Accuracy vs Interpretability  |  ResNet-50 + Grad-CAM",
                 fontsize=12, fontweight="bold")
    ax.grid(alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "accuracy_vs_interpretability.png"),
                dpi=150, bbox_inches="tight")
    plt.close()


# ─────────────────────────────────────────────────────────────────────────────
#  GRAD-CAM
# ─────────────────────────────────────────────────────────────────────────────

class GradCAM:
    def __init__(self, model: keras.Model, layer_name: str = GRADCAM_LAYER) -> None:
        self.model      = model
        self.layer_name = self._resolve_layer(model, layer_name)
        conv_layer      = model.get_layer(self.layer_name)
        self.grad_model = keras.Model(
            inputs  = model.inputs,
            outputs = [conv_layer.output, model.output])
        print(f"Grad-CAM layer: {self.layer_name}  shape={conv_layer.output.shape}")

    @staticmethod
    def _resolve_layer(model: keras.Model, preferred: str) -> str:
        layer_names = {l.name for l in model.layers}
        if preferred in layer_names:
            return preferred
        for layer in model.layers:
            if hasattr(layer, "layers"):
                if preferred in {sl.name for sl in layer.layers}:
                    return preferred
        for layer in reversed(model.layers):
            if isinstance(layer, layers.Conv2D):
                warnings.warn(f"[GradCAM] '{preferred}' not found; using '{layer.name}'.")
                return layer.name
        raise ValueError("[GradCAM] No Conv2D layer found.")

    @tf.function
    def _compute_grads(self, x_tensor):
        with tf.GradientTape() as tape:
            conv_outputs, logits = self.grad_model(x_tensor, training=False)
            tape.watch(conv_outputs)
            score = logits[:, 0]
        grads = tape.gradient(score, conv_outputs)
        return conv_outputs, grads

    def explain(self, x_img: np.ndarray,
                target_class: int = 1) -> Tuple[np.ndarray, np.ndarray]:
        x = np.asarray(x_img, dtype=np.float32)
        if x.ndim == 3:
            x = x[np.newaxis]
        if float(x.max()) > 1.0 + 1e-3:
            x = x / 255.0

        conv_outputs, grads = self._compute_grads(tf.constant(x))
        conv_outputs = conv_outputs[0].numpy()
        grads        = grads[0].numpy()

        if target_class == 0:
            grads = -grads

        pooled_grads = grads.mean(axis=(0, 1))
        cam = np.einsum("hwc,c->hw", conv_outputs, pooled_grads)
        cam = np.maximum(cam, 0.0)

        H, W    = x_img.shape[:2]
        raw_cam = cv2.resize(cam, (W, H), interpolation=cv2.INTER_LINEAR)
        cam_max = raw_cam.max()
        if cam_max > SMALL:
            raw_cam = raw_cam / cam_max

        alpha_vis = self._postprocess(raw_cam)
        return alpha_vis, raw_cam

    @staticmethod
    def _postprocess(cam: np.ndarray) -> np.ndarray:
        alpha = cam.copy().astype(np.float32)
        if alpha.max() < SMALL:
            return np.zeros_like(alpha)

        vmax = np.percentile(alpha, 99.0)
        if vmax < SMALL:
            vmax = alpha.max()
        alpha = np.clip(alpha, 0.0, vmax) / (vmax + SMALL)

        vis_sigma = max(GRADCAM_SIGMA, 5.0)
        ks = max(3, int(np.ceil(6.0 * vis_sigma))) | 1
        alpha = cv2.GaussianBlur(alpha, (ks, ks), sigmaX=vis_sigma, sigmaY=vis_sigma)

        amax = alpha.max()
        if amax > SMALL:
            alpha = alpha / amax

        if GRADCAM_POWER != 1.0:
            alpha = np.power(alpha, GRADCAM_POWER)

        amin, amax = alpha.min(), alpha.max()
        if amax - amin > SMALL:
            alpha = (alpha - amin) / (amax - amin)

        return alpha.astype(np.float32)


# ─────────────────────────────────────────────────────────────────────────────
#  VISUALISATION HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def overlay_heatmap(img_rgb: np.ndarray, alpha: np.ndarray,
                    strength: float = 0.65) -> np.ndarray:
    cmap  = plt.get_cmap("jet")
    rgba  = cmap(alpha).astype(np.float32)
    rgba[..., 3] = 0.28 * strength + (strength - 0.28 * strength) * alpha
    heat  = rgba[..., :3]
    a_map = rgba[..., 3:4]
    return np.clip(a_map * heat + (1.0 - a_map) * img_rgb, 0.0, 1.0).astype(np.float32)


# ─────────────────────────────────────────────────────────────────────────────
#  EXPLANATION PANELS
# ─────────────────────────────────────────────────────────────────────────────

def _render_class_panel(samples: List[Dict], class_label: str,
                         header_color: str, save_path: str) -> None:
    n = len(samples)
    if n == 0:
        return

    BG        = "#F7F9FC"
    CARD_BG   = "#FFFFFF"
    TITLE_COL = "#1C2B3A"
    SUB_COL   = "#4A5568"
    ACCENT    = header_color

    fig = plt.figure(figsize=(13, 4.4 * n + 1.1))
    fig.patch.set_facecolor(BG)
    gs  = GridSpec(n + 1, 3, figure=fig,
                   height_ratios=[0.12] + [1.0] * n,
                   hspace=0.08, wspace=0.06)

    col_labels = ["Original X-Ray", "Grad-CAM Heatmap", "Heatmap Overlay"]
    for ci, txt in enumerate(col_labels):
        ax = fig.add_subplot(gs[0, ci])
        ax.set_facecolor(BG)
        ax.axhline(y=0.18, xmin=0.1, xmax=0.9, color=ACCENT,
                   linewidth=3, solid_capstyle="round")
        ax.text(0.5, 0.65, txt, ha="center", va="center",
                color=TITLE_COL, fontsize=12, fontweight="bold",
                transform=ax.transAxes)
        ax.text(0.5, 0.30, class_label, ha="center", va="center",
                color=ACCENT, fontsize=10, fontweight="bold",
                transform=ax.transAxes)
        ax.axis("off")

    for row, s in enumerate(samples):
        img        = s["img"]
        alpha_disp = s["alpha"]
        prob       = s["prob"]
        pred       = "PNEUMONIA" if prob >= 0.5 else "NORMAL"
        correct    = class_label == pred
        ok_symbol  = "✓" if correct else "✗"
        ok_color   = "#2E7D32" if correct else "#C62828"
        title_str  = f"GT: {class_label}   Pred: {pred}   p(PNEU): {prob:.3f}"

        ax0 = fig.add_subplot(gs[row + 1, 0])
        ax0.set_facecolor(CARD_BG)
        ax0.imshow(img, interpolation="bilinear")
        ax0.axis("off")
        ax0.set_title(title_str, color=TITLE_COL, fontsize=7.5, pad=4, loc="center")
        ax0.text(0.97, 0.97, ok_symbol, transform=ax0.transAxes,
                 ha="right", va="top", fontsize=14, fontweight="bold",
                 color=ok_color,
                 bbox=dict(boxstyle="round,pad=0.2", fc="white",
                           ec=ok_color, lw=1.5, alpha=0.85))

        ax1 = fig.add_subplot(gs[row + 1, 1])
        ax1.set_facecolor(CARD_BG)
        im = ax1.imshow(alpha_disp, cmap="jet", vmin=0.0, vmax=1.0,
                        interpolation="bilinear", aspect="auto")
        ax1.axis("off")
        ax1.set_title("Grad-CAM", color=SUB_COL, fontsize=7.5, pad=4)
        cbar = fig.colorbar(im, ax=ax1, fraction=0.040, pad=0.02,
                            orientation="vertical")
        cbar.set_ticks([0.0, 0.5, 1.0])
        cbar.set_ticklabels(["Low", "Mid", "High"])
        cbar.ax.yaxis.set_tick_params(color=SUB_COL, labelsize=6)
        plt.setp(cbar.ax.yaxis.get_ticklabels(), color=SUB_COL)
        cbar.outline.set_edgecolor("#CCCCCC")

        ax2 = fig.add_subplot(gs[row + 1, 2])
        ax2.set_facecolor(CARD_BG)
        ax2.imshow(overlay_heatmap(img, alpha_disp, strength=0.65),
                   interpolation="bilinear")
        ax2.axis("off")
        ax2.set_title("Overlay", color=SUB_COL, fontsize=7.5, pad=4)

    fig.suptitle(
        f"{class_label}  |  ResNet-50 + Grad-CAM",
        fontsize=12, fontweight="bold", color=TITLE_COL, y=1.002)
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"Panel saved  →  {save_path}")


def _render_single_sample(s: Dict, class_label: str,
                            header_color: str, save_path: str) -> None:
    BG        = "#F7F9FC"
    CARD_BG   = "#FFFFFF"
    TITLE_COL = "#1C2B3A"
    SUB_COL   = "#4A5568"

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
    fig.patch.set_facecolor(BG)
    for ax in axes:
        ax.set_facecolor(CARD_BG)

    img        = s["img"]
    alpha_disp = s["alpha"]
    prob       = s["prob"]
    pred       = "PNEUMONIA" if prob >= 0.5 else "NORMAL"
    correct    = class_label == pred
    ok_symbol  = "✓" if correct else "✗"
    ok_color   = "#2E7D32" if correct else "#C62828"
    title_str  = f"GT: {class_label}   Pred: {pred}   p(PNEU): {prob:.3f}"

    axes[0].imshow(img, interpolation="bilinear")
    axes[0].axis("off")
    axes[0].set_title(f"Original X-Ray\n{title_str}", color=TITLE_COL, fontsize=9, pad=4)
    axes[0].text(0.97, 0.97, ok_symbol, transform=axes[0].transAxes,
                 ha="right", va="top", fontsize=15, fontweight="bold", color=ok_color,
                 bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=ok_color, lw=1.5, alpha=0.85))

    im = axes[1].imshow(alpha_disp, cmap="jet", vmin=0.0, vmax=1.0,
                        interpolation="bilinear", aspect="auto")
    axes[1].axis("off")
    axes[1].set_title("Grad-CAM Heatmap", color=SUB_COL, fontsize=9, pad=4)
    cbar = fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    cbar.set_ticks([0.0, 0.5, 1.0])
    cbar.set_ticklabels(["Low", "Mid", "High"])
    cbar.ax.yaxis.set_tick_params(color=SUB_COL, labelsize=7)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=SUB_COL)
    cbar.outline.set_edgecolor("#CCCCCC")

    axes[2].imshow(overlay_heatmap(img, alpha_disp, strength=0.65), interpolation="bilinear")
    axes[2].axis("off")
    axes[2].set_title("Heatmap Overlay", color=SUB_COL, fontsize=9, pad=4)

    fig.suptitle(
        f"{class_label}  |  ResNet-50 + Grad-CAM",
        color=TITLE_COL, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"Single sample  →  {save_path}")


def run_gradcam_analysis(test_gen, gradcam: GradCAM,
                          all_probs: np.ndarray,
                          opt_thr: float,
                          n_per_class: int = 6) -> None:
    test_gen.reset()
    panel_normal: List[Dict]    = []
    panel_pneumonia: List[Dict] = []
    sample_idx = 0

    for batch_x, batch_y in test_gen:
        for i in range(len(batch_x)):
            if sample_idx >= len(all_probs):
                break
            true_class = int(batch_y[i])
            prob       = float(all_probs[sample_idx])
            sample_idx += 1

            need = ((true_class == 0 and len(panel_normal)    < n_per_class) or
                    (true_class == 1 and len(panel_pneumonia) < n_per_class))
            if not need:
                continue

            x_img      = batch_x[i]
            pred_class = 1 if prob >= opt_thr else 0
            alpha_vis, _ = gradcam.explain(x_img, target_class=pred_class)

            entry = dict(title=f"#{sample_idx - 1}", img=x_img,
                         alpha=alpha_vis, prob=prob, true_class=true_class)
            if true_class == 0:
                panel_normal.append(entry)
            else:
                panel_pneumonia.append(entry)

        if (len(panel_normal) >= n_per_class and
                len(panel_pneumonia) >= n_per_class):
            break

    _render_class_panel(panel_normal, "NORMAL", "limegreen",
                        os.path.join(OUTPUT_DIR, "explanation_normal_panel.png"))
    _render_class_panel(panel_pneumonia, "PNEUMONIA", "#FF69B4",
                        os.path.join(OUTPUT_DIR, "explanation_pneumonia_panel.png"))

    for idx, s in enumerate(panel_normal[:5]):
        _render_single_sample(
            s, "NORMAL", "limegreen",
            os.path.join(OUTPUT_DIR, f"single_normal_{idx + 1}.png"))

    for idx, s in enumerate(panel_pneumonia[:5]):
        _render_single_sample(
            s, "PNEUMONIA", "#FF69B4",
            os.path.join(OUTPUT_DIR, f"single_pneumonia_{idx + 1}.png"))


# ─────────────────────────────────────────────────────────────────────────────
#  MRS COMPUTATION
# ─────────────────────────────────────────────────────────────────────────────

def compute_mrs(test_gen, gradcam: GradCAM,
                all_probs: np.ndarray, opt_thr: float) -> Dict:
    print(f"\n{'='*60}\nCOMPUTING MRS\n{'='*60}")
    test_gen.reset()
    records: List[Dict] = []
    sample_idx = 0
    n_failed   = 0

    for batch_x, batch_y in test_gen:
        for i in range(len(batch_x)):
            if sample_idx >= len(all_probs):
                break
            true_class = int(batch_y[i])
            prob       = float(all_probs[sample_idx])
            x_img      = batch_x[i]
            sample_idx += 1
            pred_class = 1 if prob >= opt_thr else 0

            try:
                _, raw_cam = gradcam.explain(x_img, target_class=pred_class)
            except Exception as exc:
                print(f"  WARNING sample {sample_idx}: {exc}")
                n_failed += 1
                continue

            Ri, lung_cov = compute_ri(x_img, raw_cam)
            records.append(dict(
                sample_id        = sample_idx - 1,
                true_class       = true_class,
                prob             = prob,
                pred_class       = pred_class,
                Ri               = Ri,
                lung_pixel_ratio = lung_cov,
                cam_max          = float(raw_cam.max())))

            if sample_idx % 50 == 0:
                running = np.mean([r["Ri"] for r in records])
                ok_flag = "✓" if LUNG_ADAPTIVE_LO <= lung_cov <= LUNG_ADAPTIVE_HI else "⚠"
                print(f"  [{sample_idx:>4}/{test_gen.n}]  MRS={running:.4f}  "
                      f"Ri={Ri:.4f}  cov={lung_cov:.2f}{ok_flag}")

        if sample_idx >= len(all_probs):
            break

    if not records:
        print("ERROR: No MRS records produced.")
        return {}

    all_ri  = np.array([r["Ri"] for r in records])
    ri_pneu = np.array([r["Ri"] for r in records if r["true_class"] == 1])
    ri_norm = np.array([r["Ri"] for r in records if r["true_class"] == 0])
    all_cov = np.array([r["lung_pixel_ratio"] for r in records])

    MRS_overall   = float(all_ri.mean())
    MRS_pneumonia = float(ri_pneu.mean()) if len(ri_pneu) else 0.0
    MRS_normal    = float(ri_norm.mean()) if len(ri_norm) else 0.0
    MRS_std       = float(all_ri.std())
    delta         = MRS_overall - MRS_BASELINE

    print(f"\n{'='*60}\nMEAN RELEVANCE SCORE\n{'='*60}")
    print(f"  MRS Overall   : {MRS_overall:.4f}  ±{MRS_std:.4f}")
    print(f"  MRS PNEUMONIA : {MRS_pneumonia:.4f}  (n={len(ri_pneu)})")
    print(f"  MRS NORMAL    : {MRS_normal:.4f}  (n={len(ri_norm)})")
    print(f"  Avg coverage  : {all_cov.mean():.3f}")
    print(f"  Δ vs baseline : {'+' if delta >= 0 else ''}{delta:.4f}")
    if n_failed:
        print(f"  Failed samples: {n_failed}")

    csv_path = os.path.join(OUTPUT_DIR, "mrs_per_sample.csv")
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=records[0].keys())
        writer.writeheader()
        writer.writerows(records)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].hist(all_ri, bins=30, color="steelblue", edgecolor="white", alpha=0.85)
    axes[0].axvline(MRS_overall,  color="tomato", lw=2, label=f"MRS={MRS_overall:.4f}")
    axes[0].axvline(MRS_BASELINE, color="orange", lw=1.5, linestyle="--",
                    label=f"Baseline={MRS_BASELINE:.2f}")
    axes[0].set_xlabel("Ri"); axes[0].set_ylabel("Count")
    axes[0].set_title("MRS — All Test Samples")
    axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

    axes[1].hist(ri_pneu, bins=20, color="#FF69B4", edgecolor="white", alpha=0.75,
                 label=f"PNEUMONIA  {MRS_pneumonia:.4f}")
    axes[1].hist(ri_norm, bins=20, color="limegreen", edgecolor="white", alpha=0.75,
                 label=f"NORMAL  {MRS_normal:.4f}")
    axes[1].axvline(MRS_BASELINE, color="orange", lw=1.5, linestyle="--",
                    label=f"Baseline={MRS_BASELINE:.2f}")
    axes[1].set_xlabel("Ri"); axes[1].set_ylabel("Count")
    axes[1].set_title("MRS — Per Class")
    axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

    plt.suptitle(f"MRS Histogram  |  ResNet-50 + Grad-CAM  |  MRS={MRS_overall:.4f}",
                 fontweight="bold", fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "mrs_histogram.png"), dpi=150, bbox_inches="tight")
    plt.close()

    return dict(
        MRS_overall         = round(MRS_overall,   4),
        MRS_pneumonia       = round(MRS_pneumonia, 4),
        MRS_normal          = round(MRS_normal,    4),
        MRS_std             = round(MRS_std,        4),
        n_samples_mrs       = len(records),
        n_failed_gradcam    = n_failed,
        baseline_lrp_colin  = MRS_BASELINE,
        delta_vs_baseline   = round(delta, 4),
        avg_lung_coverage   = round(float(all_cov.mean()), 4))


# ─────────────────────────────────────────────────────────────────────────────
#  ZIP ALL OUTPUTS
# ─────────────────────────────────────────────────────────────────────────────

def zip_outputs(output_dir: str, zip_name: str = "gradcam_results.zip") -> str:
    zip_path = os.path.join(output_dir, zip_name)
    image_exts = {".png", ".json", ".csv"}
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fname in sorted(os.listdir(output_dir)):
            if fname == zip_name:
                continue
            ext = os.path.splitext(fname)[1].lower()
            if ext in image_exts:
                fpath = os.path.join(output_dir, fname)
                zf.write(fpath, arcname=fname)
    print(f"\nAll outputs zipped  →  {zip_path}")
    return zip_path


# ─────────────────────────────────────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main() -> None:
    print(f"ResNet-50 + Grad-CAM + MRS  |  EPOCHS={EPOCHS}  LR={LR}  L2={L2_VALUE}")

    train_gen, val_gen, test_gen, _, steps = get_loaders(DATA_ROOT)
    class_weight_dict = compute_class_weights_from_gen(train_gen)

    ckpt = os.path.join(OUTPUT_DIR, "best_model.keras")
    if os.path.exists(ckpt):
        print(f"Loading checkpoint  ←  {ckpt}")
        model   = keras.models.load_model(ckpt)
        history = None
    else:
        model = build_resnet50()
        model.summary(line_length=90)
        model, history = train_model(model, train_gen, val_gen, steps, class_weight_dict)
        plot_training_accuracy(history)
        plot_training_loss(history)

    metrics, opt_thr = evaluate_model(model, val_gen, test_gen)

    test_gen.reset()
    all_probs = tf.sigmoid(
        model.predict(test_gen, verbose=1).ravel()
    ).numpy()[:test_gen.n]

    gradcam = GradCAM(model, layer_name=GRADCAM_LAYER)

    test_gen.reset()
    run_gradcam_analysis(test_gen, gradcam, all_probs, opt_thr, n_per_class=6)

    test_gen.reset()
    mrs_results = compute_mrs(test_gen, gradcam, all_probs, opt_thr)

    mrs_overall = mrs_results.get("MRS_overall", 0.0)
    plot_metrics_bar_chart(metrics, mrs_overall)
    plot_ppv_npv(metrics)
    plot_confusion_matrix(metrics)

    csv_path = os.path.join(OUTPUT_DIR, "mrs_per_sample.csv")
    if os.path.exists(csv_path):
        records_list = pd.read_csv(csv_path).to_dict("records")
        plot_accuracy_vs_interpretability(records_list, opt_thr)

    final = {
        **{k: v for k, v in metrics.items() if k not in ("TP","TN","FP","FN")},
        **mrs_results,
        "interpretability_technique" : "Grad-CAM",
        "gradcam_target_layer"       : GRADCAM_LAYER,
        "gradcam_reference"          : (
            "Selvaraju et al., Grad-CAM, IJCV 128, 336-359 (2020)"),
        "mrs_formula"                : (
            "Ri = Σ(CAM_pos · M) / (Σ CAM_pos + ε)  [Hedström et al. JMLR 2023]"),
        "threshold_source"           : "validation set only",
        "epochs_config"              : EPOCHS,
        "lr"                         : LR,
        "l2"                         : L2_VALUE,
    }
    with open(os.path.join(OUTPUT_DIR, "final_report.json"), "w") as f:
        json.dump({k: round(float(v), 4) if isinstance(v, float) else v
                   for k, v in final.items()}, f, indent=2)

    avg_cov = mrs_results.get("avg_lung_coverage", 0.0)
    delta   = mrs_results.get("delta_vs_baseline", 0.0)
    print(f"\n{'─'*50}  POST-RUN")
    if avg_cov < LUNG_ADAPTIVE_LO:
        print(f"⚠  cov={avg_cov:.3f} < floor — decrease lung padding params")
    elif avg_cov > LUNG_ADAPTIVE_HI:
        print(f"⚠  cov={avg_cov:.3f} > ceil — increase LUNG_BOTTOM_PAD")
    else:
        print(f"✓  cov={avg_cov:.3f}  in [{LUNG_ADAPTIVE_LO}, {LUNG_ADAPTIVE_HI}]")

    if   mrs_overall >= MRS_BASELINE:
        print(f"✓  MRS={mrs_overall:.4f} ≥ {MRS_BASELINE}  TARGET ACHIEVED")
    elif mrs_overall >= 0.80:
        print(f"△  MRS={mrs_overall:.4f}  — review coverage or CAM layer")
    else:
        print(f"✗  MRS={mrs_overall:.4f}  — check checkpoint & CAM quality")
    print(f"   Δ vs baseline ({MRS_BASELINE}): {'+' if delta >= 0 else ''}{delta:.4f}")

    zip_outputs(OUTPUT_DIR)
    print(f"\nAll outputs  →  {OUTPUT_DIR}/")


if __name__ == "__main__":
    main()

2026-05-19 02:17:42.254768: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779157062.471834      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779157062.540395      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779157063.060936      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779157063.060989      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779157063.060992      57 computation_placer.cc:177] computation placer alr

ResNet-50 + Grad-CAM + MRS  |  EPOCHS=50  LR=0.001  L2=0.0001
Found 4708 validated image filenames belonging to 2 classes.
Found 524 validated image filenames belonging to 2 classes.
Found 624 validated image filenames belonging to 2 classes.
Train=4708  Val=524  Test=624
Class map: {'NORMAL': 0, 'PNEUMONIA': 1}


I0000 00:00:1779157103.777652      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779157103.784902      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "ResNet50_Pneumonia"

...

 Trainable params: 23,536,641 (89.79 MB)

 Non-trainable params: 53,120 (207.50 KB)

Epoch 1/50


I0000 00:00:1779157149.175991     132 service.cc:152] XLA service 0x7aa340003000 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779157149.176050     132 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1779157149.176058     132 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1779157155.515528     132 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1779157181.870028     132 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


148/148 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8003 - auc: 0.8577 - loss: 1.0568
Epoch 1: val_loss improved from inf to 7.72991, saving model to ./resnet50_gradcam_output/best_model.keras
148/148 ━━━━━━━━━━━━━━━━━━━━ 311s 2s/step - accuracy: 0.8007 - auc: 0.8581 - loss: 1.0560 - val_accuracy: 0.7424 - val_auc: 0.5000 - val_loss: 7.7299 - learning_rate: 0.0010
Epoch 2/50
148/148 ━━━━━━━━━━━━━━━━━━━━ 0s 911ms/step - accuracy: 0.8959 - auc: 0.9332 - loss: 0.8434
Epoch 2: val_loss improved from 7.72991 to 5.18333, saving model to ./resnet50_gradcam_output/best_model.keras
148/148 ━━━━━━━━━━━━━━━━━━━━ 146s 988ms/step - accuracy: 0.8959 - auc: 0.9333 - loss: 0.8432 - val_accuracy: 0.2576 - val_auc: 0.5000 - val_loss: 5.1833 - learning_rate: 0.0010
Epoch 3/50
148/148 ━━━━━━━━━━━━━━━━━━━━ 0s 921ms/step - accuracy: 0.9229 - auc: 0.9521 - loss: 0.7204
Epoch 3: val_loss did not improve from 5.18333
148/148 ━━━━━━━━━━━━━━━━━━━━ 146s 988ms/step - accuracy: 0.9229 - auc: 0.9521 - loss: 0.7203

In [4]:
"""
Post-hoc statistical tests for XAI method comparison.
Reads mrs_per_sample.csv from each model variant — no retraining needed.
Requires: pip install scipy statsmodels pandas numpy
"""
!pip install scipy statsmodels pandas numpy
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from statsmodels.stats.contingency_tables import mcnemar
from itertools import combinations

# ══════════════════════════════════════════════════════════
#  CONFIGURE — point each key to its mrs_per_sample.csv
# ══════════════════════════════════════════════════════════

CSV_PATHS = {
    "LRP":          "/kaggle/input/datasets/mradyankhan/lyer-wise/mrs_per_sample.csv",
    "Grad-CAM":     "/kaggle/input/datasets/mradyankhan/grad-cam/mrs_per_sample.csv",
    "Adversarial":  "/kaggle/input/datasets/mradyankhan/adversarial/mrs_per_sample.csv",
    "SAM":          "/kaggle/input/datasets/mradyankhan/sample-attention/mrs_per_sample.csv",
}

BONFERRONI_ALPHA = 0.05          # uncorrected alpha
N_PAIRS          = 6             # C(4,2) = 6 pairs
ALPHA_CORRECTED  = BONFERRONI_ALPHA / N_PAIRS   # 0.0083

# ══════════════════════════════════════════════════════════
#  LOAD DATA
# ══════════════════════════════════════════════════════════

data = {}
for method, path in CSV_PATHS.items():
    df = pd.read_csv(path)
    # Validate expected columns are present
    for col in ["sample_id", "true_class", "pred_class", "Ri"]:
        if col not in df.columns:
            raise ValueError(f"Missing column '{col}' in {path}")
    data[method] = df.sort_values("sample_id").reset_index(drop=True)
    print(f"Loaded {method}: {len(df)} samples | "
          f"MRS={df['Ri'].mean():.4f} ± {df['Ri'].std():.4f}")

# Confirm all CSVs cover the same 624 samples
sample_ids = [set(df["sample_id"]) for df in data.values()]
assert all(s == sample_ids[0] for s in sample_ids), \
    "ERROR: CSVs have different sample_ids — they must all cover the same test set."
print(f"\nAll {len(sample_ids[0])} sample IDs match across methods. ✓\n")

methods = list(data.keys())
pairs   = list(combinations(methods, 2))   # 6 pairs

# ══════════════════════════════════════════════════════════
#  TEST 1 — WILCOXON SIGNED-RANK (Rᵢ distributions)
#  Bonferroni-corrected significance level: 0.05 / 6 = 0.0083
# ══════════════════════════════════════════════════════════

print("=" * 65)
print("WILCOXON SIGNED-RANK TEST  —  pairwise Rᵢ comparison")
print(f"Bonferroni-corrected α = {ALPHA_CORRECTED:.4f}  ({N_PAIRS} pairs)")
print("=" * 65)

wilcoxon_results = []
for m1, m2 in pairs:
    ri1 = data[m1]["Ri"].values
    ri2 = data[m2]["Ri"].values

    # alternative='two-sided': tests whether median difference ≠ 0
    stat, p_val = wilcoxon(ri1, ri2, alternative="two-sided", zero_method="wilcox")

    significant = p_val < ALPHA_CORRECTED
    direction   = (f"{m1} > {m2}" if ri1.mean() > ri2.mean()
                   else f"{m2} > {m1}")

    wilcoxon_results.append({
        "Pair":        f"{m1} vs {m2}",
        "W-statistic": round(stat, 2),
        "p-value":     round(p_val, 6),
        "Significant": "Yes ✓" if significant else "No",
        "Direction":   direction if significant else "n.s.",
    })
    print(f"  {m1:14} vs {m2:14} | "
          f"W={stat:>9.2f}  p={p_val:.6f}  "
          f"{'✓ sig' if significant else '— n.s.'}")

print()

# ══════════════════════════════════════════════════════════
#  TEST 2 — McNEMAR'S TEST (per-sample accuracy)
#  Tests whether two methods make the SAME errors
#  b = M1 correct, M2 wrong | c = M1 wrong, M2 correct
# ══════════════════════════════════════════════════════════

print("=" * 65)
print("McNEMAR'S TEST  —  pairwise per-sample accuracy")
print(f"Bonferroni-corrected α = {ALPHA_CORRECTED:.4f}  ({N_PAIRS} pairs)")
print("=" * 65)

mcnemar_results = []
for m1, m2 in pairs:
    correct1 = (data[m1]["pred_class"] == data[m1]["true_class"]).astype(int).values
    correct2 = (data[m2]["pred_class"] == data[m2]["true_class"]).astype(int).values

    # Build 2×2 contingency table
    # [0,0]=both wrong  [0,1]=M1 wrong M2 right
    # [1,0]=M1 right M2 wrong  [1,1]=both right
    n00 = int(((correct1 == 0) & (correct2 == 0)).sum())
    n01 = int(((correct1 == 0) & (correct2 == 1)).sum())   # b
    n10 = int(((correct1 == 1) & (correct2 == 0)).sum())   # c
    n11 = int(((correct1 == 1) & (correct2 == 1)).sum())
    table = [[n00, n01], [n10, n11]]

    result = mcnemar(table, exact=False, correction=True)   # continuity-corrected chi-sq
    p_val  = result.pvalue
    stat   = result.statistic
    significant = p_val < ALPHA_CORRECTED

    acc1 = correct1.mean(); acc2 = correct2.mean()
    direction = (f"{m1} better" if acc1 > acc2 else f"{m2} better")

    mcnemar_results.append({
        "Pair":        f"{m1} vs {m2}",
        "b (M1✓ M2✗)": n10,
        "c (M1✗ M2✓)": n01,
        "χ²":          round(stat, 4),
        "p-value":     round(p_val, 6),
        "Significant": "Yes ✓" if significant else "No",
        "Direction":   direction if significant else "n.s.",
    })
    print(f"  {m1:14} vs {m2:14} | "
          f"b={n10:>3}  c={n01:>3}  χ²={stat:>7.4f}  p={p_val:.6f}  "
          f"{'✓ sig' if significant else '— n.s.'}")

print()

# ══════════════════════════════════════════════════════════
#  SAVE RESULTS TO CSV
# ══════════════════════════════════════════════════════════

pd.DataFrame(wilcoxon_results).to_csv("wilcoxon_results.csv", index=False)
pd.DataFrame(mcnemar_results).to_csv("mcnemar_results.csv",  index=False)

print("Results saved:")
print("  → wilcoxon_results.csv")
print("  → mcnemar_results.csv")

# ══════════════════════════════════════════════════════════
#  SUMMARY TABLE
# ══════════════════════════════════════════════════════════

print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)
print(f"Bonferroni-corrected threshold: α_c = {ALPHA_CORRECTED:.4f}")
sig_w = sum(1 for r in wilcoxon_results if "Yes" in r["Significant"])
sig_m = sum(1 for r in mcnemar_results  if "Yes" in r["Significant"])
print(f"Wilcoxon: {sig_w}/{N_PAIRS} pairs significantly different in MRS")
print(f"McNemar:  {sig_m}/{N_PAIRS} pairs significantly different in accuracy")

Loaded LRP: 624 samples | MRS=0.9078 ± 0.0680
Loaded Grad-CAM: 624 samples | MRS=0.7963 ± 0.0868
Loaded Adversarial: 624 samples | MRS=0.8238 ± 0.0732
Loaded SAM: 624 samples | MRS=0.6681 ± 0.1105

All 624 sample IDs match across methods. ✓

WILCOXON SIGNED-RANK TEST  —  pairwise Rᵢ comparison
Bonferroni-corrected α = 0.0083  (6 pairs)
  LRP            vs Grad-CAM       | W= 10104.00  p=0.000000  ✓ sig
  LRP            vs Adversarial    | W= 17222.00  p=0.000000  ✓ sig
  LRP            vs SAM            | W=  3821.00  p=0.000000  ✓ sig
  Grad-CAM       vs Adversarial    | W= 70238.00  p=0.000000  ✓ sig
  Grad-CAM       vs SAM            | W= 21766.00  p=0.000000  ✓ sig
  Adversarial    vs SAM            | W=  8777.00  p=0.000000  ✓ sig

McNEMAR'S TEST  —  pairwise per-sample accuracy
Bonferroni-corrected α = 0.0083  (6 pairs)
  LRP            vs Grad-CAM       | b= 17  c= 39  χ²= 7.8750  p=0.005012  ✓ sig
  LRP            vs Adversarial    | b= 21  c= 29  χ²= 0.9800  p=0.322199  — n.s.